# VisionSetil E20b — Lepiota fine-tune from best.pt

Resume MAP@3 checkpoint · hard-neg subincarnata/cristata · 12 epochs · lower LR.
Orientation only · never consumption · product_unlock=false.


In [ ]:
# ═══ CELL 1: Install deps + CUDA precheck (BEFORE import torch) ═══
import sys, os, warnings, subprocess
warnings.filterwarnings('ignore')
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn'])


def _cuda_works_in_subprocess():
    test_code = (
        "import torch; "
        "x = torch.randn(8, 8, device='cuda'); "
        "_ = (x @ x.T).sum().item(); "
        "print('CUDA_OK')"
    )
    try:
        result = subprocess.run(
            [sys.executable, '-c', test_code],
            capture_output=True, text=True, timeout=60,
        )
        return result.returncode == 0 and 'CUDA_OK' in result.stdout
    except Exception:
        return False


CUDA_PRECHECK = _cuda_works_in_subprocess()

if not CUDA_PRECHECK:
    _has_gpu = (
        os.path.exists('/dev/nvidia0')
        or os.environ.get('NVIDIA_VISIBLE_DEVICES') is not None
        or os.environ.get('CUDA_VISIBLE_DEVICES') is not None
    )
    if _has_gpu:
        print("GPU detected but PyTorch CUDA kernels broken. Reinstalling...", flush=True)
        subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q',
                               'torch', 'torchvision', 'torchaudio', 'triton'],
                              stderr=subprocess.DEVNULL)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                               'torch==2.5.1', 'torchvision==0.20.1',
                               '--index-url', 'https://download.pytorch.org/whl/cu121'])
        CUDA_PRECHECK = _cuda_works_in_subprocess()
        print(f"CUDA after reinstall: {CUDA_PRECHECK}", flush=True)
    else:
        print("No GPU detected. CPU mode.", flush=True)
else:
    print("Pre-installed PyTorch CUDA works.", flush=True)

print(f"Dependencies installed. CUDA ready: {CUDA_PRECHECK}", flush=True)

In [ ]:
# ═══ CELL 2: Environment + CUDA smoke test ═══
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from pathlib import Path
import json, math, random, copy, time, subprocess, sys
from datetime import datetime
from dataclasses import dataclass, field
from collections import defaultdict
from PIL import Image

print(f"PyTorch: {torch.__version__}", flush=True)
print(f"CUDA available: {torch.cuda.is_available()}", flush=True)

CUDA_WORKS = False
if torch.cuda.is_available():
    try:
        _test = torch.randn(8, 8, device='cuda')
        _result = (_test @ _test.T).sum().item()
        CUDA_WORKS = True
        print(f"✓ CUDA smoke test PASSED. GPU: {torch.cuda.get_device_name(0)}", flush=True)
        print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB", flush=True)
    except RuntimeError as e:
        print(f"✗ CUDA smoke test FAILED: {e}", flush=True)

DEVICE = torch.device('cuda' if CUDA_WORKS else 'cpu')
N_GPU = int(torch.cuda.device_count()) if CUDA_WORKS else 0
NUM_WORKERS = 4 if CUDA_WORKS else 2
if CUDA_WORKS:
    for _gi in range(N_GPU):
        print(f"  GPU{_gi}: {torch.cuda.get_device_name(_gi)} ({torch.cuda.get_device_properties(_gi).total_memory / 1e9:.1f} GB)", flush=True)
    print(f"N_GPU={N_GPU} (T4x2 DataParallel when >=2)", flush=True)

import timm
from sklearn.metrics import f1_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
print(f"timm: {timm.__version__}", flush=True)


def log(msg, level="INFO"):
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}] [{level}] {msg}", flush=True)
    sys.stdout.flush()

log("Environment ready.")

In [ ]:
# ═══ CELL 3: Direct dataset detection (BUG 1 FIX: zero filesystem scanning) ═══
# v8 FIX: v7 spent 49 minutes on rglob. We construct known paths directly.

def detect_all_datasets():
    """Detect datasets using direct path construction — NO rglob, NO scanning.
    Handles nested Kaggle mounts: /kaggle/input/datasets/<owner>/<dataset>/
    """
    datasets = {}
    input_dir = Path('/kaggle/input')

    if not input_dir.exists():
        log("WARNING: /kaggle/input does not exist!")
        return datasets

    # ── Direct path patterns (instant, no scanning) ────────────────────────────
    # These are the KNOWN structures from Kaggle dataset mounts
    FUNGITASTIC_PATHS = [
        '/kaggle/input/datasets/picekl/fungitastic',
        '/kaggle/input/datasets/picekl',
        '/kaggle/input/fungitastic',
        '/kaggle/input/picekl',
    ]
    FUNGICLEF_PATHS = [
        # keep for future real CSV packs (seemshukla is checkpoint-only)
        '/kaggle/input/datasets/seemshukla/fungiclef',
        '/kaggle/input/datasets/seemshukla',
        '/kaggle/input/fungiclef',
        '/kaggle/input/seemshukla',
        '/kaggle/input/fungiclef2022',
        '/kaggle/input/datasets/fungiclef2022',
    ]
    # E20: GBIF ES allowlist40 — pure TEST domain
    GBIF_ES_PATHS = [
        '/kaggle/input/datasets/alonsoalviraaaa/visionsetil-gbif-es-allowlist40',
        '/kaggle/input/alonsoalviraaaa/visionsetil-gbif-es-allowlist40',
        '/kaggle/input/visionsetil-gbif-es-allowlist40',
        '/kaggle/input/datasets/alonsoalviraaaa',
    ]
    # Optional soft non-GBIF train packs
    MUSH215_PATHS = [
        '/kaggle/input/datasets/daniilonishchenko/mushrooms-images-classification-215',
        '/kaggle/input/daniilonishchenko/mushrooms-images-classification-215',
        '/kaggle/input/mushrooms-images-classification-215',
    ]

    # Try FungiTastic
    for p in FUNGITASTIC_PATHS:
        if Path(p).exists():
            datasets['fungitastic'] = Path(p)
            log(f"  ✓ Found FungiTastic: {p}")
            break

    # Try FungiCLEF / DF20 pack (optional train domain)
    for p in FUNGICLEF_PATHS:
        if Path(p).exists():
            datasets['fungiclef'] = Path(p)
            log(f"  ✓ Found FungiCLEF path: {p}")
            break

    # E20 REQUIRED test domain: GBIF ES allowlist40
    for p in GBIF_ES_PATHS:
        if Path(p).exists():
            cand = Path(p)
            if (cand / 'obs_gbif_es.jsonl').exists() or (cand / 'images').exists() or cand.name == 'visionsetil-gbif-es-allowlist40':
                datasets['gbif_es'] = cand
                log(f"  ✓ Found gbif_es (TEST domain): {p}")
                break
            for sub in cand.iterdir() if cand.is_dir() else []:
                if sub.is_dir() and ('gbif' in sub.name.lower() or (sub / 'obs_gbif_es.jsonl').exists()):
                    datasets['gbif_es'] = sub
                    log(f"  ✓ Found gbif_es nested (TEST domain): {sub}")
                    break
            if 'gbif_es' in datasets:
                break

    for p in MUSH215_PATHS:
        if Path(p).exists():
            datasets['mush215'] = Path(p)
            log(f"  ✓ Found mush215 (optional train soft): {p}")
            break

    # ── Fallback: check top-level dirs (fast, no recursion) ────────────────────
    if not datasets:
        log("Known paths not found. Checking top-level dirs...")
        for d in sorted(input_dir.iterdir()):
            if not d.is_dir():
                continue
            name = d.name.lower()
            parent = d.parent.name.lower() if d.parent != input_dir else ''
            combined = f"{parent}/{name}"

            if 'fungitastic' in combined or 'picekl' in combined:
                datasets['fungitastic'] = d
                log(f"  ✓ Found FungiTastic: {d}")
            elif 'fungiclef' in combined or 'seemshukla' in combined:
                datasets['fungiclef'] = d
                log(f"  ✓ Found FungiCLEF: {d}")
            elif 'gbif' in combined or 'visionsetil-gbif' in combined:
                datasets['gbif_es'] = d
                log(f"  ✓ Found gbif_es: {d}")
            elif 'mushrooms-images-classification-215' in combined or 'daniilonishchenko' in combined:
                datasets['mush215'] = d
                log(f"  ✓ Found mush215: {d}")

        # Check one level deep (datasets/ subfolder)
        if not datasets:
            datasets_subdir = input_dir / 'datasets'
            if datasets_subdir.exists():
                for d in sorted(datasets_subdir.iterdir()):
                    if not d.is_dir():
                        continue
                    name = d.name.lower()
                    if 'fungitastic' in name or 'picekl' in name:
                        datasets['fungitastic'] = d
                        log(f"  ✓ Found FungiTastic (nested): {d}")
                    elif 'fungiclef' in name or 'seemshukla' in name:
                        datasets['fungiclef'] = d
                        log(f"  ✓ Found FungiCLEF (nested): {d}")
                    elif 'gbif' in name or 'visionsetil-gbif' in name:
                        datasets['gbif_es'] = d
                        log(f"  ✓ Found gbif_es (nested): {d}")
                    elif '215' in name or 'daniilonishchenko' in name:
                        datasets['mush215'] = d
                        log(f"  ✓ Found mush215 (nested): {d}")

    log(f"Total datasets detected: {len(datasets)}")
    for name, path in datasets.items():
        log(f"  {name}: {path}")

    return datasets


ALL_DATASETS = detect_all_datasets()

if not ALL_DATASETS:
    log("ERROR: No datasets found! Using synthetic smoke test data.")
    ALL_DATASETS = {'synthetic': Path('/tmp/fake_data')}
    ALL_DATASETS['synthetic'].mkdir(exist_ok=True)

In [ ]:
# ═══ CELL 4: E20 multi-source loader + near-dup + split helpers ═══
# Train domain: FungiTastic (+ soft packs). Test domain: GBIF ES pure.
# Orientation only — never consumption permission.

import json
from pathlib import Path
from typing import Callable, Iterable, Optional
import pandas as pd
LogFn = Callable[[str], None]
KNOWN_JSONL_REL_PATHS = ('obs_gbif_es.jsonl', 'obs_gbif.jsonl', 'manifest.jsonl', 'observations.jsonl', 'metadata/obs_gbif_es.jsonl')
SPECIES_ALIASES = ('species', 'scientificname', 'scientific_name', 'taxon_name', 'taxon', 'expected_taxon', 'class_name', 'label', 'category', 'speciesname', 'species_name')
TAXONOMIC_RANK_COLS = frozenset({'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'specificepithet'})
IMAGE_ALIASES = ('image_path', 'imagepath', 'filename', 'file_name', 'filepath', 'file_path', 'image', 'img_path', 'img', 'image_path_jpg', 'filename_jpg', 'imageuniqueid', 'image_unique_id', 'imageid', 'image_id', 'photo_id', 'photoid')
OBS_ALIASES = ('observation_id', 'observationid', 'observation_uuid', 'observationuuid', 'obs_id', 'obsid', 'eventid', 'event_id')
SKIP_CSV_KEYWORDS = frozenset({'climatic', 'timeseries', 'climate', 'weather', 'bioclim', 'submission', 'sample_submission'})
KNOWN_CSV_REL_PATHS = ('metadata/FungiTastic/FungiTastic-ClosedSet-Train.csv', 'metadata/FungiTastic/FungiTastic-ClosedSet-Val.csv', 'metadata/FungiTastic/FungiTastic-ClosedSet-Test.csv', 'metadata/FungiTastic/FungiTastic-OpenSet-Train.csv', 'metadata/FungiTastic/FungiTastic-OpenSet-Val.csv', 'metadata/FungiTastic/FungiTastic-OpenSet-Test.csv', 'metadata/FungiTastic/FungiTastic-FewShot(train).csv', 'metadata/FungiTastic/FungiTastic-FewShot-Train.csv', 'metadata/FungiTastic/FungiTastic-FewShot/Train.csv', 'FungiTastic-FewShot/train.csv', 'FungiTastic-FewShot/Train.csv', 'DF20-train_metadata.csv', 'DF20-val_metadata.csv', 'DF20-train_metadata_PROD.csv', 'FungiCLEF2022_train_metadata.csv', 'FungiCLEF2022_test_metadata.csv', 'FungiCLEF2023_train.csv', 'metadata/FungiCLEF2023_train.csv', 'metadata/DF20-train_metadata.csv', 'train.csv', 'Train/train.csv', 'data/train.csv')
KNOWN_IMAGE_SUBDIRS = ('', 'images', 'Images', 'merged_dataset', 'train', 'Train', 'val', 'Val', 'test', 'Test', 'DF20-300px/DF20_300', 'DF20_300', 'DF20-300px', 'images/FungiTastic-FewShot/train/300p', 'images/FungiTastic-FewShot/train/500p', 'images/FungiTastic-FewShot/val/300p', 'images/FungiTastic-FewShot/val/500p', 'images/FungiTastic-FewShot/test/300p', 'images/FungiTastic-FewShot/test/500p', 'images/FungiTastic-ClosedSet/train/300p', 'images/FungiTastic-ClosedSet/train/500p', 'images/FungiTastic-ClosedSet/val/300p', 'images/FungiTastic-ClosedSet/val/500p', 'images/FungiTastic-ClosedSet/test/300p', 'images/FungiTastic-ClosedSet/test/500p', 'images/FungiTastic-OpenSet/train/300p', 'images/FungiTastic-OpenSet/train/500p', 'images/FungiTastic-OpenSet/val/300p', 'images/FungiTastic-OpenSet/val/500p', 'FungiTastic-FewShot/Train', 'FungiTastic-FewShot/Val', 'Processed_300px/JPG', 'Train/Processed_300px/JPG')
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff', '.JPG', '.JPEG', '.PNG'}

def _log(msg: str, log: Optional[LogFn]=None) -> None:
    if log:
        log(msg)
    else:
        print(msg)

def _norm_col(name: str) -> str:
    return str(name).strip().lower().replace(' ', '_')

def pick_column(columns: Iterable[str], aliases: Iterable[str]) -> Optional[str]:
    cols = list(columns)
    lower_map = {_norm_col(c): c for c in cols}
    for alias in aliases:
        key = _norm_col(alias)
        if key in lower_map:
            return lower_map[key]
    return None

def is_valid_image_csv(csv_path: Path, nrows: int=5) -> bool:
    name_lower = csv_path.name.lower()
    for kw in SKIP_CSV_KEYWORDS:
        if kw in name_lower:
            return False
    try:
        probe = pd.read_csv(csv_path, nrows=nrows)
    except Exception:
        return False
    if len(probe.columns) > 80:
        return False
    cols_l = {_norm_col(c) for c in probe.columns}
    has_species = bool(cols_l & set(SPECIES_ALIASES)) or 'species' in cols_l
    has_image = bool(cols_l & set(IMAGE_ALIASES)) or 'filename' in cols_l
    if 'imageuniqueid' in cols_l and ('scientificname' in cols_l or 'class_id' in cols_l):
        return True
    if has_species and has_image:
        return True
    if has_species and ('observationid' in cols_l or 'observation_id' in cols_l):
        return True
    return False

def find_metadata_csvs(root: Path, log: Optional[LogFn]=None) -> list[Path]:
    root = Path(root)
    found: list[Path] = []
    seen: set[str] = set()

    def _add(p: Path) -> None:
        key = str(p.resolve()) if p.exists() else str(p)
        if key in seen:
            return
        if p.exists() and p.is_file() and is_valid_image_csv(p):
            seen.add(key)
            found.append(p)
    for rel in KNOWN_CSV_REL_PATHS:
        _add(root / rel)
    for pattern in ('*.csv', 'metadata/*.csv', 'metadata/*/*.csv', 'metadata/FungiTastic/*.csv', 'Train/*.csv', 'data/*.csv'):
        try:
            for m in list(root.glob(pattern))[:30]:
                _add(m)
        except Exception:
            continue
    _log(f'  n_csv found: {len(found)}', log)
    for p in found:
        _log(f'    - {(p.relative_to(root) if p.is_relative_to(root) else p)}', log)
    return found

def _looks_binomial(value: str) -> bool:
    if not value or not isinstance(value, str):
        return False
    s = value.strip()
    parts = s.split()
    if len(parts) < 2:
        return False
    if parts[0].lower() in TAXONOMIC_RANK_COLS:
        return False
    return parts[0][:1].isupper() and parts[1][:1].islower()

def normalize_species_name(value) -> str:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return 'unknown'
    s = str(value).strip()
    if not s or s.lower() in {'nan', 'none', 'null'}:
        return 'unknown'
    s = s.replace('_', ' ').replace('-', ' ')
    s = ' '.join(s.split())
    if '(' in s:
        head = s.split('(')[0].strip()
        parts = head.split()
        if len(parts) >= 2:
            return f'{parts[0]} {parts[1]}'
        if head:
            return head
    parts = s.split()
    if len(parts) >= 2 and parts[0][0].isupper():
        return f'{parts[0]} {parts[1]}'
    return s

def normalize_columns(df: pd.DataFrame, log: Optional[LogFn]=None) -> pd.DataFrame:
    df = df.copy()
    cols = list(df.columns)
    sp_col = pick_column(cols, SPECIES_ALIASES)
    if sp_col is not None and _norm_col(sp_col) in TAXONOMIC_RANK_COLS:
        sample = df[sp_col].dropna().astype(str).head(20).tolist()
        if not any((_looks_binomial(normalize_species_name(v)) for v in sample)):
            _log(f"  skip rank-like species col '{sp_col}' (not binomial)", log)
            sp_col = None
            sp_col = pick_column([c for c in cols if _norm_col(c) not in TAXONOMIC_RANK_COLS], SPECIES_ALIASES)
    if sp_col is None:
        sp_col = pick_column(cols, ('class_id', 'category_id'))
        if sp_col is not None:
            _log(f'  WARNING: using {sp_col} as species (numeric id — may need mapping)', log)
    if sp_col is not None and _norm_col(sp_col) in TAXONOMIC_RANK_COLS - {'class_id'}:
        sample = df[sp_col].dropna().astype(str).head(20).tolist()
        if not any((_looks_binomial(str(v)) for v in sample)):
            _log(f"  refusing taxonomic rank col '{sp_col}' as species label", log)
            sp_col = None
    img_col = pick_column(cols, IMAGE_ALIASES)
    obs_col = pick_column(cols, OBS_ALIASES)
    rename: dict[str, str] = {}
    if sp_col and sp_col != 'species' and ('species' not in df.columns):
        rename[sp_col] = 'species'
    if img_col and img_col != 'image_path' and ('image_path' not in df.columns):
        rename[img_col] = 'image_path'
    if obs_col and obs_col != 'observation_id' and ('observation_id' not in df.columns):
        rename[obs_col] = 'observation_id'
    if rename:
        _log(f'  column map: {rename}', log)
        df = df.rename(columns=rename)
    sci = pick_column(df.columns, ('scientificname', 'scientific_name'))
    if 'species' not in df.columns and sci:
        df['species'] = df[sci].map(normalize_species_name)
    elif 'species' in df.columns:
        df['species'] = df['species'].map(normalize_species_name)
        if sci and sci != 'species':
            mask = df['species'].isin(['unknown', ''])
            if mask.any():
                df.loc[mask, 'species'] = df.loc[mask, sci].map(normalize_species_name)
    if 'observation_id' not in df.columns:
        if 'image_path' in df.columns:
            df['observation_id'] = df['image_path'].astype(str).map(lambda p: Path(p).stem.split('_')[0].split('-')[-1] if p else 'unk')
        else:
            df['observation_id'] = range(len(df))
    if 'species' not in df.columns:
        if 'image_path' in df.columns:
            df['species'] = df['image_path'].astype(str).map(lambda p: normalize_species_name(Path(p).parent.name))
            bad = ~df['species'].map(_looks_binomial)
            if bad.any():
                df.loc[bad, 'species'] = 'unknown'
        else:
            df['species'] = 'unknown'
    if 'genus' not in df.columns:
        df['genus'] = df['species'].astype(str).str.split().str[0]
    for col in ('family', 'habitat', 'substrate', 'smell', 'country'):
        if col not in df.columns:
            alt = pick_column(df.columns, (col, col.capitalize(), col.upper()))
            if alt and alt != col:
                df[col] = df[alt]
            else:
                df[col] = 'unknown'
    return df

def build_filename_index(root: Path, max_files: int=150000, log: Optional[LogFn]=None) -> dict[str, str]:
    root = Path(root)
    index: dict[str, str] = {}
    n = 0
    for sub in KNOWN_IMAGE_SUBDIRS:
        d = root / sub if sub else root
        if not d.exists() or not d.is_dir():
            continue
        try:
            if sub in ('',) or sub.count('/') == 0:
                iterator = d.rglob('*') if sub.lower() in {'images', 'merged_dataset', 'train', 'val', 'test'} else d.iterdir()
            else:
                iterator = d.iterdir()
            for p in iterator:
                if not p.is_file():
                    continue
                if p.suffix not in IMAGE_EXTS and p.suffix.lower() not in {e.lower() for e in IMAGE_EXTS}:
                    continue
                key = p.name.lower()
                if key not in index:
                    index[key] = str(p.resolve())
                    n += 1
                    if n >= max_files:
                        _log(f'  filename index capped at {max_files}', log)
                        return index
        except Exception as e:
            _log(f'  index skip {d}: {e}', log)
            continue
    images_root = root / 'images'
    if images_root.exists():
        try:
            for p in images_root.rglob('*'):
                if not p.is_file():
                    continue
                if p.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}:
                    continue
                key = p.name.lower()
                if key not in index:
                    index[key] = str(p.resolve())
                    n += 1
                    if n >= max_files:
                        break
        except Exception as e:
            _log(f'  images rglob limited: {e}', log)
    _log(f'  filename index size: {len(index)}', log)
    return index

def resolve_one_image_path(raw, root: Path, index: Optional[dict[str, str]]=None) -> str:
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return ''
    s = str(raw).strip().replace(chr(92), '/')
    if not s:
        return ''
    p = Path(s)
    if p.is_absolute() and p.exists():
        return str(p)
    cand = root / s
    if cand.exists():
        return str(cand.resolve())
    name = p.name
    if index:
        hit = index.get(name.lower())
        if hit:
            return hit
    for sub in KNOWN_IMAGE_SUBDIRS:
        c = root / sub / name if sub else root / name
        if c.exists():
            return str(c.resolve())
    for split in ('train', 'val', 'test'):
        for res in ('300p', '500p', 'full', '700p'):
            for pack in ('FungiTastic-FewShot', 'FungiTastic-ClosedSet', 'FungiTastic-OpenSet'):
                c = root / 'images' / pack / split / res / name
                if c.exists():
                    return str(c.resolve())
    for sub in ('DF20-300px/DF20_300', 'DF20_300', 'DF20-300px'):
        c = root / sub / name
        if c.exists():
            return str(c.resolve())
    return str(root / s)

def resolve_image_paths(df: pd.DataFrame, root: Path, index: Optional[dict[str, str]]=None, log: Optional[LogFn]=None, drop_missing: bool=False) -> pd.DataFrame:
    if 'image_path' not in df.columns:
        return df
    root = Path(root)
    if index is None:
        index = build_filename_index(root, log=log)
    df = df.copy()
    df['image_path'] = df['image_path'].map(lambda r: resolve_one_image_path(r, root, index))
    if drop_missing:
        before = len(df)
        df = df[df['image_path'].map(lambda p: Path(str(p)).exists())].reset_index(drop=True)
        _log(f'  drop_missing images: {before} -> {len(df)}', log)
    return df

def _looks_species_dirname(name: str) -> bool:
    s = name.strip()
    if not s or s.lower() in {'metadata', 'images', 'train', 'val', 'test', 'merged_dataset', 'data', 'captions', 'climaticdata', '__pycache__'}:
        return False
    if ' ' in s:
        return _looks_binomial(normalize_species_name(s))
    if '_' in s or '-' in s:
        return _looks_binomial(normalize_species_name(s))
    return False

def find_jsonl_manifests(root: Path, log: Optional[LogFn]=None) -> list[Path]:
    root = Path(root)
    found: list[Path] = []
    seen: set[str] = set()

    def _add(p: Path) -> None:
        key = str(p.resolve()) if p.exists() else str(p)
        if key in seen or not p.is_file():
            return
        try:
            with p.open(encoding='utf-8') as f:
                for _ in range(5):
                    line = f.readline()
                    if not line.strip():
                        continue
                    obj = json.loads(line)
                    if not isinstance(obj, dict):
                        return
                    keys = {str(k).lower() for k in obj.keys()}
                    has_sp = bool(keys & {'species', 'scientificname', 'scientific_name'})
                    has_img = bool(keys & {'image_paths', 'image_path', 'filename', 'filepath', 'file_path'})
                    if has_sp and has_img:
                        seen.add(key)
                        found.append(p)
                    return
        except Exception:
            return
    for rel in KNOWN_JSONL_REL_PATHS:
        _add(root / rel)
    for pattern in ('*.jsonl', 'metadata/*.jsonl'):
        try:
            for m in list(root.glob(pattern))[:20]:
                _add(m)
        except Exception:
            continue
    _log(f'  n_jsonl found: {len(found)}', log)
    for p in found:
        try:
            _log(f'    - {p.relative_to(root)}', log)
        except Exception:
            _log(f'    - {p}', log)
    return found

def dataset_kind(root: Path) -> str:
    root = Path(root)
    if not root.exists():
        return 'empty'
    has_csv = False
    has_jsonl = False
    has_tfrec = False
    has_pth = False
    has_img = False
    has_species_dirs = False
    try:
        top = list(root.iterdir())
    except Exception:
        return 'unknown'
    for p in top[:200]:
        name = p.name.lower()
        if p.is_file():
            if name.endswith('.csv'):
                has_csv = True
            elif name.endswith('.jsonl'):
                has_jsonl = True
            elif name.endswith('.tfrec') or name.endswith('.tfrecord'):
                has_tfrec = True
            elif name.endswith('.pth') or name.endswith('.pt') or name.endswith('.ckpt'):
                has_pth = True
            elif Path(name).suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}:
                has_img = True
        elif p.is_dir():
            if name in {'metadata', 'images', 'train', 'merged_dataset', 'df20-300px'}:
                has_csv = has_csv or name == 'metadata'
                has_img = has_img or name in {'images', 'train', 'merged_dataset', 'df20-300px'}
            if _looks_species_dirname(p.name):
                try:
                    if any(p.iterdir()):
                        has_species_dirs = True
                except Exception:
                    pass
    if (root / 'metadata').exists():
        has_csv = True
    if (root / 'images').exists() or (root / 'merged_dataset').exists():
        has_img = True
        img_root = root / 'images'
        if img_root.is_dir() and (not has_species_dirs):
            try:
                for sp in list(img_root.iterdir())[:40]:
                    if sp.is_dir() and _looks_species_dirname(sp.name):
                        has_species_dirs = True
                        break
            except Exception:
                pass
    if not has_jsonl:
        for rel in KNOWN_JSONL_REL_PATHS:
            if (root / rel).is_file():
                has_jsonl = True
                break
    if has_jsonl:
        return 'jsonl_manifest'
    if has_csv or (has_img and has_species_dirs):
        if has_csv:
            return 'csv_images'
        return 'folder_species'
    if has_species_dirs:
        return 'folder_species'
    if has_tfrec and (not has_csv) and (not has_img):
        return 'tfrecord_only'
    if has_pth and (not has_csv) and (not has_img):
        return 'checkpoint_only'
    return 'unknown'

def _folder_observation_id(species_norm: str, stem: str) -> str:
    import re
    base = re.sub('[_\-]\d+$', '', stem).strip()
    if not base:
        base = stem
    sp_flat = species_norm.replace(' ', '_').lower()
    base_l = base.lower().replace(' ', '_')
    if base_l.startswith(sp_flat):
        rest = base_l[len(sp_flat):].lstrip('_-')
        if rest:
            base = rest
        else:
            base = 'view'
    return f'{species_norm}::{base}'

def _resolve_manifest_path(raw: str, root: Path) -> str:
    s = str(raw).strip().replace(chr(92), '/')
    if not s:
        return ''
    cand = Path(s)
    if cand.is_absolute() and cand.exists():
        return str(cand)
    for prefix in ('data/industrial_v1/gbif/', 'industrial_v1/gbif/', 'data/industrial_v1/'):
        if s.startswith(prefix):
            s = s[len(prefix):]
            break
    candidates = [root / s]
    parts = Path(s).parts
    if parts and parts[0] != 'images' and (len(parts) >= 2):
        candidates.append(root / 'images' / Path(*parts[-2:]))
    if len(parts) >= 1:
        candidates.append(root / 'images' / parts[-1])
        candidates.append(root / parts[-1])
    if len(parts) >= 2 and parts[0] == 'images':
        candidates.append(root / Path(*parts))
    for c in candidates:
        try:
            if c.exists() and c.is_file():
                return str(c.resolve())
        except Exception:
            continue
    return ''

def load_from_jsonl_manifest(root: Path, db_name: str, log: Optional[LogFn]=None, max_images: int=120000, prefer_cc_ok: bool=True) -> pd.DataFrame:
    root = Path(root)
    manifests = find_jsonl_manifests(root, log=log)
    if not manifests:
        return pd.DataFrame()
    records: list[dict] = []
    for man in manifests:
        try:
            with man.open(encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        row = json.loads(line)
                    except json.JSONDecodeError:
                        continue
                    if not isinstance(row, dict):
                        continue
                    sp_raw = row.get('species') or row.get('scientificName') or row.get('scientific_name') or 'unknown'
                    sp = normalize_species_name(sp_raw)
                    oid = row.get('observation_id') or row.get('observationID') or row.get('obs_id')
                    lic = row.get('license_class') or row.get('licenseClass') or 'unknown'
                    paths = row.get('image_paths') or row.get('image_path') or row.get('filename')
                    if paths is None:
                        continue
                    if isinstance(paths, str):
                        paths = [paths]
                    if not isinstance(paths, (list, tuple)):
                        continue
                    for raw_p in paths:
                        if raw_p is None:
                            continue
                        s = str(raw_p).strip().replace(chr(92), '/')
                        if not s:
                            continue
                        resolved = _resolve_manifest_path(s, root)
                        if not resolved:
                            continue
                        if oid is None:
                            stem = Path(resolved).stem
                            oid_use = f"{sp}::{stem.split('_')[0]}"
                        else:
                            oid_use = str(oid)
                        records.append({'image_path': resolved, 'species': sp, 'observation_id': oid_use, 'genus': sp.split()[0] if sp else 'unknown', 'family': 'unknown', 'habitat': 'unknown', 'substrate': 'unknown', 'smell': 'unknown', 'country': row.get('country') or 'ES', 'license_class': str(lic).lower() if lic else 'unknown', 'license': row.get('license') or ''})
                        if len(records) >= max_images:
                            break
                    if len(records) >= max_images:
                        break
        except Exception as e:
            _log(f'  ERROR reading jsonl {man}: {e}', log)
        if len(records) >= max_images:
            break
    df = pd.DataFrame.from_records(records)
    if len(df) == 0:
        _log(f'  jsonl manifest: 0 existing images from {root}', log)
        return df
    if prefer_cc_ok and 'license_class' in df.columns:
        rank = df['license_class'].map(lambda x: 0 if str(x).lower() == 'cc_ok' else 1)
        df = df.assign(_lic_rank=rank).sort_values('_lic_rank').drop(columns=['_lic_rank'])

    def _prefix_oid(v: str) -> str:
        s = str(v)
        if s.startswith(f'{db_name}_') or s.startswith('gbif_'):
            return s if s.startswith(f'{db_name}_') or db_name.startswith('gbif') else f'{db_name}_{s}'
        return f'{db_name}_{s}'
    if db_name.startswith('gbif') or db_name in {'gbif_es', 'gbif'}:
        df['observation_id'] = df['observation_id'].astype(str).map(lambda s: s if str(s).startswith('gbif_') else f'gbif_{s}')
    else:
        df['observation_id'] = df['observation_id'].astype(str).map(_prefix_oid)
    df['source_db'] = db_name
    n_cc = int((df['license_class'].astype(str).str.lower() == 'cc_ok').sum()) if 'license_class' in df.columns else 0
    _log(f"  jsonl: {len(df)} images, {df['species'].nunique()} spp, {df['observation_id'].nunique()} obs, cc_ok={n_cc}", log)
    return df.reset_index(drop=True)

def load_from_folder_structure(root: Path, db_name: str, log: Optional[LogFn]=None, max_images: int=80000) -> pd.DataFrame:
    root = Path(root)
    records = []
    search_roots = [root]
    for sub in ('images', 'merged_dataset', 'train', 'Train', 'data'):
        if (root / sub).exists():
            search_roots.append(root / sub)
    for base in search_roots:
        try:
            entries = sorted([p for p in base.iterdir() if p.is_dir()])
        except Exception:
            continue
        for sp_dir in entries:
            sp_name = sp_dir.name.strip()
            if sp_name.lower() in {'metadata', 'climaticdata', 'captions', 'train', 'val', 'test', 'images', '__pycache__'}:
                continue
            sp_norm = normalize_species_name(sp_name)
            if sp_norm == 'unknown' or not _looks_binomial(sp_norm):
                if ' ' not in sp_norm:
                    continue
            try:
                files = list(sp_dir.iterdir())
            except Exception:
                continue
            for img in files:
                if not img.is_file():
                    continue
                if img.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}:
                    continue
                stem = img.stem
                obs = _folder_observation_id(sp_norm, stem)
                records.append({'image_path': str(img.resolve()), 'species': sp_norm, 'observation_id': obs, 'genus': sp_norm.split()[0] if sp_norm else 'unknown', 'family': 'unknown', 'habitat': 'unknown', 'substrate': 'unknown', 'smell': 'unknown', 'country': 'unknown'})
                if len(records) >= max_images:
                    break
            if len(records) >= max_images:
                break
        if len(records) >= max_images:
            break
    df = pd.DataFrame.from_records(records)
    if len(df):
        df['source_db'] = db_name
        df['observation_id'] = db_name + '_' + df['observation_id'].astype(str)
    _log(f"  folder structure: {len(df)} images, {(df['species'].nunique() if len(df) else 0)} spp, {(df['observation_id'].nunique() if len(df) else 0)} obs", log)
    return df

def load_csvs_from_root(root: Path, db_name: str, log: Optional[LogFn]=None, build_index: bool=True) -> pd.DataFrame:
    root = Path(root)
    csvs = find_metadata_csvs(root, log=log)
    if not csvs:
        _log(f'  WARNING: No valid image CSV found in {root}', log)
        return pd.DataFrame()
    frames = []
    for csv_path in csvs:
        try:
            part = pd.read_csv(csv_path, low_memory=False)
            _log(f'  read {csv_path.name}: shape={part.shape}', log)
            part = normalize_columns(part, log=log)
            part['_meta_csv'] = csv_path.name
            frames.append(part)
        except Exception as e:
            _log(f'  ERROR reading {csv_path}: {e}', log)
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    n_rows = len(df)
    _log(f'  n_rows after parse (pre-path): {n_rows}', log)
    index = build_filename_index(root, log=log) if build_index else {}
    df = resolve_image_paths(df, root, index=index, log=log, drop_missing=False)
    df['observation_id'] = db_name + '_' + df['observation_id'].astype(str)
    df['source_db'] = db_name
    if 'image_path' in df.columns:
        exists_mask = df['image_path'].map(lambda p: Path(str(p)).exists())
        n_ok = int(exists_mask.sum())
        n_all = len(df)
        _log(f'  paths existing on disk: {n_ok}/{n_all}', log)
        ratio = n_ok / max(n_all, 1)
        if n_ok == 0:
            _log(f"  WARNING: 0 existing images for '{db_name}' — source counts as empty for gate", log)
            return pd.DataFrame()
        if ratio < 0.05:
            _log(f"  WARNING: <5% images resolved ({ratio:.2%}) for '{db_name}' — keeping existing only; check layout vs filename", log)
        df = df.loc[exists_mask].reset_index(drop=True)
    _log(f"  Loaded CSV source '{db_name}': {len(df)} existing images, {df['species'].nunique()} spp, {df['observation_id'].nunique()} obs", log)
    return df

def load_single_dataset(root: Path, db_name: str, log: Optional[LogFn]=None) -> pd.DataFrame:
    root = Path(root)
    _log(f"Loading dataset '{db_name}' from {root}...", log)
    kind = dataset_kind(root)
    _log(f'  dataset_kind={kind}', log)
    if kind == 'checkpoint_only':
        _log(f"  FATAL-soft: '{db_name}' looks like model checkpoints only (.pth) — not image data", log)
        return pd.DataFrame()
    if kind == 'tfrecord_only':
        _log(f"  FATAL-soft: '{db_name}' is TFRecord-only without CSV labels — cannot map to allowlist species safely", log)
        return pd.DataFrame()
    df = pd.DataFrame()
    if kind == 'jsonl_manifest' or find_jsonl_manifests(root):
        _log('  Trying JSONL manifest loader...', log)
        df = load_from_jsonl_manifest(root, db_name, log=log)
    if (df is None or len(df) == 0) and kind in {'csv_images', 'unknown', 'empty', 'jsonl_manifest'}:
        df = load_csvs_from_root(root, db_name, log=log)
    if (df is None or len(df) == 0) and kind in {'folder_species', 'csv_images', 'unknown', 'jsonl_manifest'}:
        _log('  Trying folder-structure loader...', log)
        df = load_from_folder_structure(root, db_name, log=log)
    if df is None or len(df) == 0:
        _log(f"  Loaded: 0 images from '{db_name}'", log)
        return pd.DataFrame()
    if 'license_class' not in df.columns:
        df['license_class'] = 'unknown'
    _log(f"  Loaded: {len(df)} images, {df['species'].nunique()} species, {df['observation_id'].nunique()} observations", log)
    return df

def count_existing_images(df: Optional[pd.DataFrame]) -> int:
    if df is None or len(df) == 0 or 'image_path' not in df.columns:
        return 0
    return int(df['image_path'].map(lambda p: Path(str(p)).exists()).sum())

def fair_cap_observations(df: pd.DataFrame, max_obs: int=200, max_obs_deadly: int=400, deadly_force: Optional[set]=None, species_col: str='species', obs_col: str='observation_id', source_col: str='source_db', prefer_cc_ok: bool=True, license_col: str='license_class') -> pd.DataFrame:
    if df is None or len(df) == 0:
        return df if df is not None else pd.DataFrame()
    deadly = {str(s).lower() for s in deadly_force or set()}
    has_lic = prefer_cc_ok and license_col in df.columns

    def _oid_sort_key(g: pd.DataFrame, oid) -> tuple:
        sub = g[g[obs_col] == oid]
        n_img = len(sub)
        if has_lic:
            lic_penalty = 0 if (sub[license_col].astype(str).str.lower() == 'cc_ok').any() else 1
        else:
            lic_penalty = 0
        return (lic_penalty, -n_img, str(oid))
    parts = []
    for sp, group in df.groupby(species_col):
        cap = max_obs_deadly if str(sp).lower() in deadly else max_obs
        sources = sorted(group[source_col].astype(str).unique().tolist())
        n_src_sp = max(len(sources), 1)
        per_src = max(1, cap // n_src_sp)
        picked: list = []
        remaining = cap
        for sdb in sources:
            g = group[group[source_col].astype(str) == sdb]
            oids = list(g[obs_col].unique())
            oids_sorted = sorted(oids, key=lambda oid: _oid_sort_key(g, oid))
            take = min(per_src, remaining, len(oids_sorted))
            picked.extend(oids_sorted[:take])
            remaining -= take
        if remaining > 0:
            picked_set = set(picked)
            leftover = [oid for oid in group[obs_col].unique() if oid not in picked_set]
            leftover_sorted = sorted(leftover, key=lambda oid: _oid_sort_key(group, oid))
            picked.extend(leftover_sorted[:remaining])
        parts.append(group[group[obs_col].isin(picked)])
    if not parts:
        return df.iloc[0:0].copy()
    return pd.concat(parts, ignore_index=True)

def load_all_datasets(datasets: dict[str, Path], log: Optional[LogFn]=None, min_sources: int=1, hard_fail_below_min: bool=False) -> pd.DataFrame:
    frames = []
    per_source_rows: dict[str, int] = {}
    per_source_existing: dict[str, int] = {}
    for db_name, root in datasets.items():
        try:
            df_ds = load_single_dataset(Path(root), db_name, log=log)
            n_rows = len(df_ds) if df_ds is not None else 0
            n_exist = count_existing_images(df_ds)
            per_source_rows[db_name] = n_rows
            per_source_existing[db_name] = n_exist
            if n_exist > 0 and df_ds is not None and ('image_path' in df_ds.columns):
                mask = df_ds['image_path'].map(lambda p: Path(str(p)).exists())
                df_ds = df_ds.loc[mask].reset_index(drop=True)
                frames.append(df_ds)
            elif n_exist > 0 and df_ds is not None:
                frames.append(df_ds)
        except Exception as e:
            _log(f'ERROR loading {db_name}: {e}', log)
            per_source_rows[db_name] = 0
            per_source_existing[db_name] = 0
    nonzero = {k: v for k, v in per_source_existing.items() if v > 0}
    _log(f'  per-source row counts: {per_source_rows}', log)
    _log(f'  per-source existing image counts: {per_source_existing}', log)
    _log(f'  non-zero sources (existing images): {list(nonzero.keys())}', log)
    if len(nonzero) < min_sources:
        msg = f'MULTI-SOURCE GATE: expected ≥{min_sources} sources with existing images, got {len(nonzero)}: {nonzero}'
        _log(f"  {('FATAL' if hard_fail_below_min else 'WARNING')}: {msg}", log)
        if hard_fail_below_min:
            raise RuntimeError(msg)
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    _log(f"COMBINED: {len(df)} existing images, {df['species'].nunique()} species, {df['observation_id'].nunique()} observations from {len(nonzero)} DBs", log)
    return df

import re
from collections import defaultdict
from pathlib import Path
from typing import Any, Callable, Optional
import pandas as pd
LogFn = Callable[[str], None]
_SOURCE_PRIORITY = {'fungitastic': 0, 'mushroom1': 1, 'combined_mushrooms': 2, 'mush215': 3, 'fungiclef': 4, 'df20': 5, 'gbif_es': 10, 'gbif': 10}

def stem_key(path: str) -> str:
    p = Path(str(path).replace(chr(92), '/'))
    return p.stem.lower()

def media_id_key(path: str) -> Optional[str]:
    stem = stem_key(path)
    m = re.match('^(\d{6,})', stem)
    return m.group(1) if m else None

def _filesize_key(path: str) -> Optional[tuple[int, str]]:
    try:
        p = Path(str(path).replace(chr(92), '/'))
        if not p.is_file():
            return None
        sz = p.stat().st_size
        return (int(sz), stem_key(path))
    except OSError:
        return None

def _row_priority(row: pd.Series, train_sources: set[str]) -> tuple:
    lic = str(row.get('license_class', 'unknown') or 'unknown').lower()
    lic_rank = 0 if lic == 'cc_ok' else 1
    src = str(row.get('source_db', '') or '')
    if src in train_sources:
        src_rank = _SOURCE_PRIORITY.get(src, 6)
    else:
        src_rank = _SOURCE_PRIORITY.get(src, 20)
    return (lic_rank, src_rank, str(row.get('observation_id', '')))

def near_dup_collapse(df: pd.DataFrame, train_sources: Optional[set[str]]=None, path_col: str='image_path', use_filesize: bool=False, log: Optional[LogFn]=None) -> tuple[pd.DataFrame, dict[str, Any]]:
    if df is None or len(df) == 0:
        return (df if df is not None else pd.DataFrame(), {'n_in': 0, 'n_out': 0, 'n_collapsed': 0, 'n_groups': 0})
    train_sources = train_sources or {'fungitastic'}
    df = df.reset_index(drop=True).copy()
    n_in = len(df)
    parent = list(range(n_in))

    def find(x: int) -> int:
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a: int, b: int) -> None:
        ra, rb = (find(a), find(b))
        if ra != rb:
            parent[ra] = rb
    key_to_idx: dict[str, int] = {}

    def link_key(idx: int, key: str) -> None:
        if not key:
            return
        if key in key_to_idx:
            union(idx, key_to_idx[key])
        else:
            key_to_idx[key] = idx
    for i, row in df.iterrows():
        path = str(row.get(path_col, '') or '')
        if not path:
            continue
        st = stem_key(path)
        link_key(int(i), f'stem:{st}')
        mid = media_id_key(path)
        if mid:
            link_key(int(i), f'media:{mid}')
        if use_filesize:
            fsk = _filesize_key(path)
            if fsk is not None:
                link_key(int(i), f'size:{fsk[0]}:{fsk[1]}')
    groups: dict[int, list[int]] = defaultdict(list)
    for i in range(n_in):
        groups[find(i)].append(i)
    keep_indices: list[int] = []
    n_collapsed_rows = 0
    multi_member = 0
    for members in groups.values():
        if len(members) == 1:
            keep_indices.append(members[0])
            continue
        multi_member += 1
        oid_counts = df.loc[members, 'observation_id'].value_counts() if 'observation_id' in df.columns else None

        def score(idx: int) -> tuple:
            row = df.loc[idx]
            pri = _row_priority(row, train_sources)
            oid = str(row.get('observation_id', ''))
            oid_boost = -int(oid_counts.get(oid, 1)) if oid_counts is not None else 0
            return (*pri, oid_boost, idx)
        best = min(members, key=score)
        keep_indices.append(best)
        n_collapsed_rows += len(members) - 1
    keep_indices.sort()
    out = df.loc[keep_indices].reset_index(drop=True)
    stats = {'n_in': n_in, 'n_out': len(out), 'n_collapsed': n_collapsed_rows, 'n_groups': len(groups), 'n_multi_member_groups': multi_member, 'train_sources': sorted(train_sources)}
    if log:
        log(f'Near-dup collapse: {n_in} -> {len(out)} rows (collapsed {n_collapsed_rows} dups, multi-groups={multi_member})')
    return (out, stats)

def near_dup_keys_for_row(path: str, use_filesize: bool=False) -> set[str]:
    keys: set[str] = set()
    if not path:
        return keys
    st = stem_key(path)
    keys.add(f'stem:{st}')
    mid = media_id_key(path)
    if mid:
        keys.add(f'media:{mid}')
    if use_filesize:
        fsk = _filesize_key(path)
        if fsk is not None:
            keys.add(f'size:{fsk[0]}:{fsk[1]}')
    return keys

def shared_near_dup_keys(paths_a: list[str], paths_b: list[str], use_filesize: bool=False) -> set[str]:
    ka: set[str] = set()
    kb: set[str] = set()
    for p in paths_a:
        ka |= near_dup_keys_for_row(p, use_filesize=use_filesize)
    for p in paths_b:
        kb |= near_dup_keys_for_row(p, use_filesize=use_filesize)
    return ka & kb

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Optional
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
LogFn = Callable[[str], None]
TRAIN_DOMAIN_DEFAULT = frozenset({'fungitastic', 'mushroom1', 'combined_mushrooms', 'mush215', 'fungiclef', 'df20'})
TEST_DOMAIN_DEFAULT = frozenset({'gbif_es', 'gbif'})

def deadly_recall_at_k(probs: np.ndarray, labels: np.ndarray, deadly_idxs: set[int], k: int=3) -> tuple[float, int]:
    top = np.argsort(-probs, axis=1)[:, :k]
    mask = np.array([int(y) in deadly_idxs for y in labels], dtype=bool)
    n = int(mask.sum())
    if n == 0:
        return (0.0, 0)
    hits = sum((1 for i in range(len(labels)) if mask[i] and labels[i] in top[i]))
    return (float(hits / n), n)

def deadly_top1_from_preds(preds: np.ndarray, labels: np.ndarray, deadly_idxs: set[int]) -> tuple[float, int]:
    mask = np.array([int(y) in deadly_idxs for y in labels], dtype=bool)
    n = int(mask.sum())
    if n == 0:
        return (0.0, 0)
    return (float((preds[mask] == labels[mask]).mean()), n)

def deadly_gate_eval(safety_recall_deadly_at_3: float, n_deadly: int, threshold: float=0.5) -> dict[str, Any]:
    n = int(n_deadly)
    if n <= 0:
        return {'pass': False, 'status': 'unevaluable', 'n_deadly': 0, 'threshold': float(threshold), 'value': None, 'reason': 'deadly gate unevaluable: 0 deadly samples in test'}
    val = float(safety_recall_deadly_at_3)
    return {'pass': val >= float(threshold), 'status': 'ok', 'n_deadly': n, 'threshold': float(threshold), 'value': val, 'reason': None}

def drop_test_rows_sharing_near_dup_keys(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame, path_col: str='image_path', hard_fail: bool=False, log: Optional[LogFn]=None) -> tuple[pd.DataFrame, dict[str, Any]]:
    try:
        from kaggle.near_dup import near_dup_keys_for_row as _nk
        from kaggle.near_dup import shared_near_dup_keys as _sk
    except ImportError:
        _nk = globals().get('near_dup_keys_for_row')
        _sk = globals().get('shared_near_dup_keys')
        if _nk is None or _sk is None:
            raise RuntimeError('near_dup helpers unavailable for residual key scrub')
    _log = log or (lambda m: None)
    if len(test_df) == 0:
        return (test_df, {'n_shared_keys': 0, 'n_dropped_rows': 0, 'hard_fail': hard_fail})
    tv_paths = []
    if len(train_df) and path_col in train_df.columns:
        tv_paths.extend(train_df[path_col].astype(str).tolist())
    if len(val_df) and path_col in val_df.columns:
        tv_paths.extend(val_df[path_col].astype(str).tolist())
    te_paths = test_df[path_col].astype(str).tolist() if path_col in test_df.columns else []
    shared = _sk(tv_paths, te_paths, use_filesize=False)
    if not shared:
        return (test_df.reset_index(drop=True), {'n_shared_keys': 0, 'n_dropped_rows': 0, 'hard_fail': hard_fail, 'pass': True})
    if hard_fail:
        raise AssertionError(f'LEAK: {len(shared)} near-dup keys still shared train/val↔test after collapse')
    keep_mask = []
    for p in te_paths:
        keys = _nk(p, use_filesize=False)
        keep_mask.append(len(keys & shared) == 0)
    before = len(test_df)
    out = test_df.loc[keep_mask].reset_index(drop=True)
    dropped = before - len(out)
    _log(f'  Dropped {dropped} test rows sharing near-dup keys with train/val ({len(shared)} keys)')
    if len(out) == 0:
        raise RuntimeError('SOURCE HOLDOUT GATE: test emptied after near-dup residual drop')
    return (out, {'n_shared_keys': len(shared), 'n_dropped_rows': dropped, 'hard_fail': hard_fail, 'pass': True, 'action': 'drop_contaminated_test_rows'})

def assert_obs_disjoint(train_ids: set[str], val_ids: set[str], test_ids: set[str], hard_fail: bool=True) -> dict[str, Any]:
    leaks = {'train_val': len(train_ids & val_ids), 'train_test': len(train_ids & test_ids), 'val_test': len(val_ids & test_ids)}
    ok = all((v == 0 for v in leaks.values()))
    result = {'leaks': leaks, 'pass': ok}
    if hard_fail and (not ok):
        raise AssertionError(f'LEAK: observation_id overlap {leaks}')
    return result

def _obs_level_split(image_df: pd.DataFrame, val_size: float=0.15, seed: int=42, min_per_class: int=2, log: Optional[LogFn]=None) -> tuple[pd.DataFrame, pd.DataFrame]:
    if len(image_df) == 0:
        empty = image_df.iloc[0:0].copy()
        return (empty, empty)
    agg: dict[str, str] = {'species': 'first'}
    if 'genus' in image_df.columns:
        agg['genus'] = 'first'
    if 'source_db' in image_df.columns:
        agg['source_db'] = 'first'
    obs_df = image_df.groupby('observation_id').agg(agg).reset_index()
    counts = obs_df['species'].value_counts()
    valid = counts[counts >= min_per_class].index
    dropped = counts[counts < min_per_class]
    if log is not None and len(dropped):
        log(f'  train-domain min_per_class={min_per_class}: drop {len(dropped)} sparse spp')
    obs_df = obs_df[obs_df['species'].isin(valid)].copy()
    if len(obs_df) == 0:
        empty = image_df.iloc[0:0].copy()
        return (empty, empty)
    species_final = obs_df['species'].value_counts()
    large = species_final[species_final >= 4].index
    small = species_final[(species_final >= min_per_class) & (species_final < 4)].index
    obs_large = obs_df[obs_df['species'].isin(large)].copy()
    obs_small = obs_df[obs_df['species'].isin(small)].copy()
    train_parts: list[pd.DataFrame] = []
    val_parts: list[pd.DataFrame] = []
    if len(obs_large) > 0:
        try:
            tr, va = train_test_split(obs_large, test_size=val_size, random_state=seed, stratify=obs_large['species'])
        except ValueError:
            tr, va = train_test_split(obs_large, test_size=val_size, random_state=seed)
        train_parts.append(tr)
        val_parts.append(va)
    if len(obs_small) > 0:
        if len(obs_small) >= 2 and val_size > 0:
            tr, va = train_test_split(obs_small, test_size=val_size, random_state=seed)
            train_parts.append(tr)
            val_parts.append(va)
        else:
            train_parts.append(obs_small)
    train_obs = pd.concat(train_parts, ignore_index=True) if train_parts else pd.DataFrame()
    val_obs = pd.concat(val_parts, ignore_index=True) if val_parts else pd.DataFrame()
    train_ids = set(train_obs['observation_id'].astype(str)) if len(train_obs) else set()
    val_ids = set(val_obs['observation_id'].astype(str)) if len(val_obs) else set()
    train_df = image_df[image_df['observation_id'].astype(str).isin(train_ids)].reset_index(drop=True)
    val_df = image_df[image_df['observation_id'].astype(str).isin(val_ids)].reset_index(drop=True)
    return (train_df, val_df)

def source_holdout_split(df: pd.DataFrame, train_sources: Optional[set[str]]=None, test_sources: Optional[set[str]]=None, val_size: float=0.15, seed: int=42, min_per_class: int=2, require_train_core: str='fungitastic', require_test_core: str='gbif_es', hard_fail_cross_domain_oids: bool=True, log: Optional[LogFn]=None) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    train_sources = set(train_sources or TRAIN_DOMAIN_DEFAULT)
    test_sources = set(test_sources or TEST_DOMAIN_DEFAULT)
    _log = log or (lambda m: None)
    if df is None or len(df) == 0:
        raise RuntimeError('source_holdout_split: empty dataframe')
    if 'source_db' not in df.columns:
        raise RuntimeError('source_holdout_split: source_db column required')
    if 'observation_id' not in df.columns:
        raise RuntimeError('source_holdout_split: observation_id column required')
    src_counts = df['source_db'].astype(str).value_counts().to_dict()
    _log(f'Source hold-out inputs: {src_counts}')
    train_domain = df[df['source_db'].astype(str).isin(train_sources)].copy()
    test_df = df[df['source_db'].astype(str).isin(test_sources)].copy()
    if require_train_core and require_train_core not in set(train_domain['source_db'].astype(str)):
        raise RuntimeError(f"SOURCE HOLDOUT GATE: train core '{require_train_core}' missing; train_domain sources={(train_domain['source_db'].value_counts().to_dict() if len(train_domain) else {})}")
    if require_test_core and require_test_core not in set(test_df['source_db'].astype(str)):
        present_test = set(test_df['source_db'].astype(str)) if len(test_df) else set()
        if not present_test & test_sources:
            raise RuntimeError(f"SOURCE HOLDOUT GATE: test core '{require_test_core}' missing; test sources={(test_df['source_db'].value_counts().to_dict() if len(test_df) else {})}")
    if len(train_domain) == 0:
        raise RuntimeError('SOURCE HOLDOUT GATE: empty train domain')
    if len(test_df) == 0:
        raise RuntimeError('SOURCE HOLDOUT GATE: empty GBIF test domain')
    train_oids = set(train_domain['observation_id'].astype(str))
    test_oids = set(test_df['observation_id'].astype(str))
    cross = train_oids & test_oids
    if cross:
        msg = f'SOURCE HOLDOUT LEAK: {len(cross)} observation_ids in both train-domain and test-domain (examples={sorted(cross)[:5]})'
        if hard_fail_cross_domain_oids:
            raise AssertionError(msg)
        _log(f'  WARNING (soft): {msg} — dropping from test')
        test_df = test_df[~test_df['observation_id'].astype(str).isin(cross)].copy()
        if len(test_df) == 0:
            raise RuntimeError('SOURCE HOLDOUT GATE: test emptied after cross-oid drop')
    train_df, val_df = _obs_level_split(train_domain, val_size=val_size, seed=seed, min_per_class=min_per_class, log=log)
    if len(train_df) == 0:
        raise RuntimeError('SOURCE HOLDOUT GATE: empty train after val split')
    train_ids = set(train_df['observation_id'].astype(str))
    val_ids = set(val_df['observation_id'].astype(str))
    test_ids = set(test_df['observation_id'].astype(str))
    leak_info = assert_obs_disjoint(train_ids, val_ids, test_ids, hard_fail=True)
    meta: dict[str, Any] = {'protocol': 'source_holdout_e20b_lepiota_ft', 'train_sources': sorted(train_sources), 'test_sources': sorted(test_sources), 'n_train_obs': len(train_ids), 'n_val_obs': len(val_ids), 'n_test_obs': len(test_ids), 'n_train_imgs': len(train_df), 'n_val_imgs': len(val_df), 'n_test_imgs': len(test_df), 'train_source_counts': train_df['source_db'].value_counts().to_dict() if len(train_df) else {}, 'val_source_counts': val_df['source_db'].value_counts().to_dict() if len(val_df) else {}, 'test_source_counts': test_df['source_db'].value_counts().to_dict() if len(test_df) else {}, 'leaks': leak_info['leaks'], 'pass': leak_info['pass'], 'cross_domain_oids': len(cross), 'hard_fail_cross_domain_oids': hard_fail_cross_domain_oids, 'val_domain': 'train_domain_holdout', 'test_domain': 'gbif_es_only', 'primary_metrics': 'test_gbif', 'orientation_only': True, 'seed': seed, 'val_size': val_size}
    _log(f'Source hold-out split: train={len(train_ids)} obs ({len(train_df)} imgs) | val={len(val_ids)} obs ({len(val_df)} imgs) | test={len(test_ids)} obs ({len(test_df)} imgs) [GBIF pure]')
    _log(f"  train sources: {meta['train_source_counts']}")
    _log(f"  test sources:  {meta['test_source_counts']}")
    return (train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True), meta)

def _obs_export_records(image_df: pd.DataFrame) -> list[dict[str, Any]]:
    records = []
    if len(image_df) == 0:
        return records
    cols = image_df.columns
    for oid, group in image_df.groupby('observation_id'):
        paths = group['image_path'].astype(str).tolist() if 'image_path' in cols else []
        rec: dict[str, Any] = {'observation_id': str(oid), 'species': str(group['species'].iloc[0]) if 'species' in cols else 'unknown', 'source_db': str(group['source_db'].iloc[0]) if 'source_db' in cols else 'unknown', 'image_paths': paths, 'n_images': len(paths)}
        if 'license_class' in cols:
            rec['license_class'] = str(group['license_class'].iloc[0])
        if 'genus' in cols:
            rec['genus'] = str(group['genus'].iloc[0])
        records.append(rec)
    records.sort(key=lambda r: r['observation_id'])
    return records

def export_split_artifacts(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame, out_dir: Path | str, split_meta: Optional[dict[str, Any]]=None, near_dup_stats: Optional[dict[str, Any]]=None, hard_fail: bool=True) -> dict[str, Any]:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    train_ids = set(train_df['observation_id'].astype(str)) if len(train_df) else set()
    val_ids = set(val_df['observation_id'].astype(str)) if len(val_df) else set()
    test_ids = set(test_df['observation_id'].astype(str)) if len(test_df) else set()
    leak_info = assert_obs_disjoint(train_ids, val_ids, test_ids, hard_fail=hard_fail)
    train_recs = _obs_export_records(train_df)
    val_recs = _obs_export_records(val_df)
    test_recs = _obs_export_records(test_df)
    nl = chr(10)
    (out_dir / 'train_obs.json').write_text(json.dumps(train_recs, indent=2, ensure_ascii=False) + nl, encoding='utf-8')
    (out_dir / 'val_obs.json').write_text(json.dumps(val_recs, indent=2, ensure_ascii=False) + nl, encoding='utf-8')
    (out_dir / 'test_obs.json').write_text(json.dumps(test_recs, indent=2, ensure_ascii=False) + nl, encoding='utf-8')
    manifest = {'generated_at': datetime.now(timezone.utc).isoformat(), 'protocol': (split_meta or {}).get('protocol', 'source_holdout_e20b_lepiota_ft'), 'n_train_obs': len(train_recs), 'n_val_obs': len(val_recs), 'n_test_obs': len(test_recs), 'n_train_imgs': len(train_df), 'n_val_imgs': len(val_df), 'n_test_imgs': len(test_df), 'leaks': leak_info['leaks'], 'pass': leak_info['pass'], 'split_meta': split_meta or {}, 'near_dup_stats': near_dup_stats or {}, 'label_sources': {'fungitastic': 'FT metadata CSVs (species / scientificName)', 'gbif_es': 'GBIF species field in obs_gbif_es.jsonl'}, 'orientation_only': True, 'policy': 'never_consumption_permission', 'files': ['train_obs.json', 'val_obs.json', 'test_obs.json', 'split_manifest.json']}
    (out_dir / 'split_manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + nl, encoding='utf-8')
    return manifest

df = load_all_datasets(
    ALL_DATASETS,
    log=log,
    min_sources=2,
    hard_fail_below_min=True,
)
if df is None or len(df) == 0:
    log("FATAL: No data loaded after multi-source gate!")
    df = pd.DataFrame()
else:
    log(f"E20 source_db counts: {df['source_db'].value_counts().to_dict()}")
    _present = set(df['source_db'].unique().tolist()) if 'source_db' in df.columns else set()
    if 'gbif_es' in ALL_DATASETS and 'gbif_es' not in _present:
        raise RuntimeError(
            f"GBIF GATE: mount exists at {ALL_DATASETS['gbif_es']} but contributed 0 rows"
        )
    if 'fungitastic' in ALL_DATASETS and 'fungitastic' not in _present:
        log('  WARNING: fungitastic mounted but 0 rows after load')
    if 'gbif_es' not in ALL_DATASETS:
        log('  WARNING: gbif_es dataset not detected under /kaggle/input')

In [ ]:
# ═══ CELL 5: E20 industrial allowlist + near-dup collapse + domain caps ═══
# Scope lock: 40 spp until honest MAP@3≥0.22 AND deadly@3≥0.50 on GBIF hold-out.
# Product language: orientation only, never consumption permission.
ALLOWLIST = {'Lepiota cristata', 'Kuehneromyces mutabilis', 'Amanita virosa', 'Laccaria laccata', 'Armillaria lutea', 'Suillus grevillei', 'Galerina marginata', 'Hypholoma fasciculare', 'Craterellus tubaeformis', 'Fomitopsis pinicola', 'Lactarius deterrimus', 'Lepista nuda', 'Coprinus comatus', 'Chlorophyllum olivieri', 'Clitocybe nebularis', 'Pluteus cervinus', 'Boletus edulis', 'Paxillus involutus', 'Gyromitra esculenta', 'Laccaria amethystina', 'Amanita muscaria', 'Suillus luteus', 'Laetiporus sulphureus', 'Amanita citrina', 'Amanita pantherina', 'Pleurotus ostreatus', 'Lepiota subincarnata', 'Amanita rubescens', 'Marasmius oreades', 'Phallus impudicus', 'Imleria badia', 'Agaricus campestris', 'Scleroderma citrinum', 'Trametes versicolor', 'Cortinarius rubellus', 'Hygrophoropsis aurantiaca', 'Amanita phalloides', 'Russula ochroleuca', 'Leccinum scabrum', 'Lepiota castanea'}
DEADLY_FORCE = {
    'amanita phalloides', 'amanita virosa', 'amanita muscaria', 'amanita pantherina',
    'galerina marginata', 'gyromitra esculenta', 'cortinarius rubellus',
    'hypholoma fasciculare', 'lepiota castanea', 'lepiota subincarnata', 'paxillus involutus',
}
TRAIN_DOMAIN_SOURCES = {'fungitastic', 'mushroom1', 'combined_mushrooms', 'mush215', 'fungiclef', 'df20'}
TEST_DOMAIN_SOURCES = {'gbif_es', 'gbif'}
NEAR_DUP_STATS = {}

if len(df) > 0:
    df['species'] = df['species'].astype(str).str.strip()
    allow_l = {a.lower() for a in ALLOWLIST}
    df = df[df['species'].str.lower().isin(allow_l)].copy()
    if 'image_path' in df.columns:
        _ex = df['image_path'].map(lambda p: Path(str(p)).exists())
        log(f"  existing images post-allowlist (pre-cap): {int(_ex.sum())}/{len(df)}")
        df = df.loc[_ex].copy() if _ex.any() else df.iloc[0:0].copy()

    log(f"After allowlist filter: {len(df)} imgs, {df['species'].nunique() if len(df) else 0} spp")
    src_counts = df['source_db'].value_counts().to_dict() if len(df) and 'source_db' in df.columns else {}
    log(f"  Sources pre-cap: {src_counts}")
    n_src = int(df['source_db'].nunique()) if len(df) and 'source_db' in df.columns else 0
    log(f"  n_sources_after_allowlist={n_src}")
    if n_src < 2:
        msg = (
            f"MULTI-SOURCE GATE: expected ≥2 sources with rows after allowlist+existing images, "
            f"got {n_src}: {src_counts}"
        )
        log(f"  FATAL: {msg}")
        raise RuntimeError(msg)
    _have = set(src_counts.keys())
    if 'gbif_es' in ALL_DATASETS and 'gbif_es' not in _have:
        raise RuntimeError(f"GBIF GATE: 0 allowlist rows from gbif_es; sources={src_counts}")
    if 'fungitastic' in ALL_DATASETS and 'fungitastic' not in _have:
        raise RuntimeError(f"FT GATE: 0 allowlist rows from fungitastic; sources={src_counts}")

    # Near-dup collapse BEFORE domain split (stem/media; prefer cc_ok then train source)
    before_nd = len(df)
    df, NEAR_DUP_STATS = near_dup_collapse(
        df, train_sources=TRAIN_DOMAIN_SOURCES, use_filesize=False, log=log,
    )
    log(f"  Near-dup: {before_nd} → {len(df)} (collapsed {NEAR_DUP_STATS.get('n_collapsed', 0)})")

    species_counts = df.groupby('observation_id')['species'].first().str.lower().value_counts()
    keep = []
    for sp, n in species_counts.items():
        mn = 1 if str(sp).lower() in DEADLY_FORCE else 2
        if n >= mn:
            keep.append(sp)
    df = df[df['species'].str.lower().isin(keep)].copy()

    # Fair caps PER DOMAIN so GBIF cannot starve FT train pool (and vice versa)
    MAX_OBS = 200
    MAX_OBS_DEADLY = 400
    parts_capped = []
    for domain_name, domain_srcs in [('train', TRAIN_DOMAIN_SOURCES), ('test', TEST_DOMAIN_SOURCES)]:
        sub = df[df['source_db'].astype(str).isin(domain_srcs)].copy()
        if len(sub) == 0:
            log(f"  domain {domain_name}: 0 rows pre-cap")
            continue
        sub = fair_cap_observations(
            sub, max_obs=MAX_OBS, max_obs_deadly=MAX_OBS_DEADLY, deadly_force=DEADLY_FORCE,
            prefer_cc_ok=True,
        )
        log(f"  domain {domain_name} post-cap: {len(sub)} imgs, "
            f"{sub['observation_id'].nunique()} obs, sources={sub['source_db'].value_counts().to_dict()}")
        parts_capped.append(sub)
    if not parts_capped:
        raise RuntimeError("E20 GATE: no domain rows after caps")
    df = pd.concat(parts_capped, ignore_index=True)

    if 'image_path' in df.columns:
        before = len(df)
        df = df.drop_duplicates(subset=['image_path'], keep='first')
        log(f"  Dedup image_path: {before} → {len(df)}")

    src_counts3 = df['source_db'].value_counts().to_dict() if len(df) and 'source_db' in df.columns else {}
    if 'gbif_es' not in src_counts3 and 'gbif' not in src_counts3:
        raise RuntimeError(f"GBIF GATE: test domain dropped after caps: {src_counts3}")
    if 'fungitastic' not in src_counts3:
        raise RuntimeError(f"FT GATE: train domain dropped after caps: {src_counts3}")

    if 'license_class' in df.columns:
        _lc = df['license_class'].astype(str).str.lower().value_counts().to_dict()
        log(f"  license_class post-cap: {_lc}")

    DATABASES_USED_EFFECTIVE = sorted(df['source_db'].unique().tolist()) if len(df) and 'source_db' in df.columns else []
    log(f"E20 source-holdout prep: imgs={len(df)} spp={df['species'].nunique()} obs={df['observation_id'].nunique()}")
    log(f"  Source DBs: {df['source_db'].value_counts().to_dict()}")
    log(f"  databases_used: {DATABASES_USED_EFFECTIVE}")
    log("  Label sources: FT metadata CSVs; GBIF species field in JSONL")
    log("  Inputs are image-only (no species-name path features to model)")
else:
    log("WARNING: empty df after allowlist")
    DATABASES_USED_EFFECTIVE = []
    raise RuntimeError(
        "SOURCE HOLDOUT GATE: empty df after allowlist — cannot train E20"
    )

In [ ]:
# ═══ CELL 6: Auto-label view types ═══
def infer_view_type(row):
    text = str(row.get('image_path', '')).lower()
    text += ' ' + str(row.get('observation_id', '')).lower()

    if any(kw in text for kw in ['gill', 'lamina', 'underside', 'pore', 'hymenium']):
        return 'gills'
    if any(kw in text for kw in ['cap', 'pileus', 'top', 'zenit', 'above']):
        return 'detail'
    if any(kw in text for kw in ['habitat', 'context', 'env', 'situ', 'landscape', 'scene']):
        return 'habitat'
    if any(kw in text for kw in ['front', 'side', 'profile', 'stem', 'stipe', 'base', 'full']):
        return 'front'
    return None


df['view_type'] = df.apply(infer_view_type, axis=1)

VIEW_ROTATION = ['gills', 'front', 'habitat', 'detail']
for obs_id, group in df.groupby('observation_id'):
    mask = df.loc[df['observation_id'] == obs_id, 'view_type'].isna()
    unlabeled_indices = df.index[df['observation_id'] == obs_id][mask]
    for i, idx in enumerate(unlabeled_indices):
        df.loc[idx, 'view_type'] = VIEW_ROTATION[i % len(VIEW_ROTATION)]

log(f"View type distribution:\n{df['view_type'].value_counts().to_string()}")

In [ ]:
# ═══ CELL 7: E20 SOURCE HOLDOUT split + persist split artifacts ═══
# Train/val = FT domain; Test = pure GBIF ES. No mixed random test (E19 inflate).
# Assert train∩val∩test obs empty; fail hard if not.
# Cross-domain oid overlap hard-fails; residual near-dup keys drop contaminated test rows.
log("E20 source hold-out protocol (honest product gate)")

train_df, val_df, test_df, SPLIT_META = source_holdout_split(
    df,
    train_sources=TRAIN_DOMAIN_SOURCES,
    test_sources=TEST_DOMAIN_SOURCES,
    val_size=0.15,
    seed=42,
    min_per_class=2,
    require_train_core='fungitastic',
    require_test_core='gbif_es',
    hard_fail_cross_domain_oids=True,
    log=log,
)

# Residual near-dup hygiene: drop contaminated test rows (fail if test emptied)
test_df, _nd_scrub = drop_test_rows_sharing_near_dup_keys(
    train_df, val_df, test_df, hard_fail=False, log=log,
)
SPLIT_META['n_shared_near_dup_keys_post_split'] = int(_nd_scrub.get('n_shared_keys', 0))
SPLIT_META['n_test_rows_dropped_near_dup'] = int(_nd_scrub.get('n_dropped_rows', 0))
SPLIT_META['n_test_obs'] = int(test_df['observation_id'].nunique()) if len(test_df) else 0
SPLIT_META['n_test_imgs'] = int(len(test_df))
if _nd_scrub.get('n_shared_keys', 0) > 0:
    log(f"  Near-dup residual scrub: dropped {_nd_scrub.get('n_dropped_rows', 0)} test rows "
        f"({_nd_scrub.get('n_shared_keys')} shared keys)")
else:
    log("  Near-dup keys train/val↔test: empty (good)")

# Re-assert disjoint after scrub
assert_obs_disjoint(
    set(train_df['observation_id'].astype(str)),
    set(val_df['observation_id'].astype(str)),
    set(test_df['observation_id'].astype(str)),
    hard_fail=True,
)

OUT_DIR = Path('/kaggle/working/models')
OUT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_MANIFEST = export_split_artifacts(
    train_df, val_df, test_df, OUT_DIR,
    split_meta=SPLIT_META,
    near_dup_stats=globals().get('NEAR_DUP_STATS') or {},
    hard_fail=True,
)
log(f"Split artifacts saved under {OUT_DIR}: train_obs/val_obs/test_obs/split_manifest.json")
log(f"  protocol={SPLIT_MANIFEST.get('protocol')} pass={SPLIT_MANIFEST.get('pass')}")

In [ ]:
# ═══ CELL 8: Build observation-level multi-view records ═══
VIEW_TYPES = ('gills', 'front', 'habitat', 'detail')
VIEW_TO_IDX = {v: i for i, v in enumerate(VIEW_TYPES)}


def build_observation_records(image_df, max_views=10):
    records = []
    for obs_id, group in image_df.groupby('observation_id'):
        species = group['species'].iloc[0]
        genus = group['genus'].iloc[0]
        family = group['family'].iloc[0] if 'family' in group.columns else 'unknown'
        habitat = group['habitat'].iloc[0] if 'habitat' in group.columns else 'unknown'
        substrate = group['substrate'].iloc[0] if 'substrate' in group.columns else 'unknown'
        smell = group['smell'].iloc[0] if 'smell' in group.columns else 'unknown'
        country = group['country'].iloc[0] if 'country' in group.columns else 'unknown'

        images = []
        for _, row in group.head(max_views).iterrows():
            img_path = row.get('image_path', '')
            view = row.get('view_type', 'front')
            if view not in VIEW_TO_IDX:
                view = 'front'
            images.append((str(img_path), view))

        if len(images) > 0:
            records.append({
                'observation_id': str(obs_id),
                'images': images,
                'species': str(species),
                'genus': str(genus),
                'family': str(family),
                'habitat': str(habitat),
                'substrate': str(substrate),
                'smell': str(smell),
                'country': str(country),
            })
    return records


train_obs = build_observation_records(train_df)
val_obs = build_observation_records(val_df)
test_obs = build_observation_records(test_df)

log(f"Observations: train={len(train_obs)} | val={len(val_obs)} | test={len(test_obs)}")

all_species = sorted(set(r['species'] for r in train_obs + val_obs))
label2idx = {s: i for i, s in enumerate(all_species)}
idx2label = {i: s for s, i in label2idx.items()}
NUM_CLASSES = len(label2idx)
log(f"Classes: {NUM_CLASSES}")

In [ ]:
# ═══ CELL 9: Multi-View Dataset with torchvision transforms v2 ═══
from torchvision.transforms import v2 as T

class MultiViewDataset(Dataset):
    def __init__(self, observations, label2idx, image_size=224, augment=False):
        self.observations = observations
        self.label2idx = label2idx
        self.image_size = image_size
        self.augment = augment

        if augment:
            self.transform = T.Compose([
                T.ToImage(),
                T.ToDtype(torch.float32, scale=True),
                T.RandomHorizontalFlip(),
                T.RandomResizedCrop(size=(image_size, image_size), scale=(0.7, 1.0), antialias=True),
                T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
                T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ])
        else:
            self.transform = T.Compose([
                T.ToImage(),
                T.ToDtype(torch.float32, scale=True),
                T.Resize(size=(image_size, image_size), antialias=True),
                T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ])

    def __len__(self):
        return len(self.observations)

    def _resolve_image_path(self, raw_path):
        p = Path(str(raw_path))
        if p.exists():
            return str(p)
        for db_name, root in ALL_DATASETS.items():
            try:
                joined = root / raw_path
                if joined.exists():
                    return str(joined)
            except Exception:
                pass
            name = p.name
            for sub in ['', 'Train', 'train', 'Test', 'test', 'images', 'merged_dataset',
                         'Train/Processed_300px/JPG', 'val', 'Val',
                         'FungiTastic-FewShot/Train', 'FungiTastic-FewShot/Val',
                         'images/FungiTastic-FewShot/train/300p',
                         'images/FungiTastic-FewShot/train/500p',
                         'images/FungiTastic-FewShot/val/300p',
                         'images/FungiTastic-FewShot/val/500p',
                         'images/FungiTastic-FewShot/test/300p',
                         'images/FungiTastic-FewShot/test/500p',
                         'FungiTastic-FewShot/Train/Processed_300px/JPG',
                         'FungiTastic-FewShot/Val/Processed_500px/JPG',
                         'DF20-300px/DF20_300', 'DF20_300',
                         'metadata/FungiTastic']:
                candidate = root / sub / name
                if candidate.exists():
                    return str(candidate)
        return str(raw_path)

    def _load_image(self, path):
        path = self._resolve_image_path(path)
        try:
            img = Image.open(path).convert('RGB')
            tensor = self.transform(img)
        except Exception:
            tensor = torch.randn(3, self.image_size, self.image_size) * 0.1
        return tensor

    def __getitem__(self, idx):
        obs = self.observations[idx]
        images = []
        view_indices = []
        for img_path, view_type in obs['images']:
            img = self._load_image(img_path)
            images.append(img)
            view_indices.append(VIEW_TO_IDX.get(view_type, 1))
        images_tensor = torch.stack(images)
        view_tensor = torch.tensor(view_indices, dtype=torch.long)
        label = self.label2idx.get(obs['species'], 0)
        return {
            'images': images_tensor,
            'view_idx': view_tensor,
            'label': torch.tensor(label, dtype=torch.long),
            'observation_id': obs['observation_id'],
            'metadata': {
                'habitat': obs['habitat'],
                'substrate': obs['substrate'],
                'smell': obs['smell'],
                'country': obs['country'],
            },
        }


def collate_fn(batch):
    max_views = max(len(obs['images']) for obs in batch)
    B = len(batch)
    H, W = batch[0]['images'].size(-2), batch[0]['images'].size(-1)
    images_padded = torch.zeros(B, max_views, 3, H, W)
    view_idx_padded = torch.zeros(B, max_views, dtype=torch.long)
    attention_mask = torch.zeros(B, max_views, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    observation_ids = []
    metadata_raw = {'habitat': [], 'substrate': [], 'smell': [], 'country': []}
    for i, obs in enumerate(batch):
        n = len(obs['images'])
        images_padded[i, :n] = obs['images']
        view_idx_padded[i, :n] = obs['view_idx']
        attention_mask[i, :n] = True
        labels[i] = obs['label']
        observation_ids.append(obs['observation_id'])
        for k in metadata_raw:
            metadata_raw[k].append(obs['metadata'][k])
    return {
        'images': images_padded,
        'view_idx': view_idx_padded,
        'attention_mask': attention_mask,
        'labels': labels,
        'observation_ids': observation_ids,
        'metadata_raw': metadata_raw,
    }


log("MultiViewDataset + collate_fn defined.")

In [ ]:
# ═══ CELL 10: Vectorized LoRA Adapter ═══
class VectorizedLoRA(nn.Module):
    def __init__(self, in_features, num_views=4, rank=16, alpha=16.0):
        super().__init__()
        self.in_features = in_features
        self.num_views = num_views
        self.rank = rank
        self.scaling = alpha / rank
        self.lora_A = nn.Parameter(torch.randn(num_views, rank, in_features) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(num_views, in_features, rank))
        for v in range(num_views):
            nn.init.kaiming_uniform_(self.lora_A.data[v].unsqueeze(0), a=math.sqrt(5))

    def forward(self, features, view_idx):
        A = self.lora_A[view_idx]
        B = self.lora_B[view_idx]
        x = features.unsqueeze(-1)
        hidden = torch.bmm(A, x)
        delta = torch.bmm(B, hidden).squeeze(-1)
        return features + delta * self.scaling


log("VectorizedLoRA defined.")

In [ ]:
# ═══ CELL 11: View-Conditioned Backbone (tiny + vectorized LoRA + safe scatter) ═══
class ViewConditionedBackbone(nn.Module):
    def __init__(self, backbone_name='convnextv2_tiny.fcmae_ft_in22k_in1k',
                 d_model=512, lora_rank=16, num_views=4):
        super().__init__()
        try:
            self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0)
        except Exception:
            log(f"WARNING: {backbone_name} not available, falling back to convnext_tiny")
            self.backbone = timm.create_model('convnext_tiny', pretrained=True, num_classes=0)

        feat_dim = self.backbone.num_features
        self.feat_dim = feat_dim
        self.d_model = d_model
        self.lora = VectorizedLoRA(feat_dim, num_views=num_views, rank=lora_rank)
        self.view_embed = nn.Embedding(num_views, feat_dim)
        self.proj = nn.Sequential(
            nn.Linear(feat_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
        )

    def forward(self, images, view_idx, attention_mask=None):
        B, N, C, H, W = images.shape
        if attention_mask is None:
            attention_mask = torch.ones(B, N, dtype=torch.bool, device=images.device)

        real_mask = attention_mask.reshape(-1)
        flat_images = images.reshape(-1, C, H, W)
        real_images = flat_images[real_mask]

        if real_images.size(0) > 0:
            real_features = self.backbone(real_images)
            features = torch.zeros(B * N, self.feat_dim, device=images.device)
            real_indices = torch.where(real_mask)[0]
            features = features.index_copy(0, real_indices, real_features)
        else:
            features = torch.zeros(B * N, self.feat_dim, device=images.device)

        flat_view = view_idx.reshape(-1).clamp(0, self.lora.num_views - 1)
        features = self.lora(features, flat_view)
        view_emb = self.view_embed(flat_view)
        features = features + view_emb
        features = features * real_mask.unsqueeze(-1).float()
        features = features.view(B, N, self.feat_dim)
        embeddings = self.proj(features)
        embeddings = embeddings * attention_mask.unsqueeze(-1).float()
        return embeddings


log("ViewConditionedBackbone defined.")

In [ ]:
# ═══ CELL 12: Metadata Encoder ═══
class MetadataEncoder(nn.Module):
    def __init__(self, vocab_sizes=None, embed_dim=32, out_dim=64):
        super().__init__()
        vocab_sizes = vocab_sizes or {'habitat': 100, 'substrate': 50, 'smell': 30, 'country': 200}
        self.embeddings = nn.ModuleDict({
            name: nn.Embedding(size, embed_dim)
            for name, size in vocab_sizes.items()
        })
        total_dim = embed_dim * len(vocab_sizes)
        self.mlp = nn.Sequential(
            nn.Linear(total_dim, out_dim * 2),
            nn.LayerNorm(out_dim * 2),
            nn.GELU(),
            nn.Linear(out_dim * 2, out_dim),
        )

    def forward(self, metadata_indices):
        embeds = []
        for name in ['habitat', 'substrate', 'smell', 'country']:
            idx = metadata_indices.get(name, torch.tensor([0], device=DEVICE))
            embeds.append(self.embeddings[name](idx))
        concat = torch.cat(embeds, dim=-1)
        return self.mlp(concat)


log("MetadataEncoder defined.")

In [ ]:
# ═══ CELL 13: Attention Fusion ═══
class AttentionFusion(nn.Module):
    def __init__(self, d_model=512, metadata_dim=64, num_heads=4, max_views=10):
        super().__init__()
        self.d_model = d_model
        self.metadata_dim = metadata_dim
        self.max_views = max_views
        self.meta_proj = nn.Linear(metadata_dim, d_model)
        self.view_pos = nn.Embedding(max_views + 1, d_model)
        self.self_attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=num_heads, batch_first=True
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Linear(d_model * 2, d_model),
        )
        self.output_dim = d_model + metadata_dim

    def forward(self, visual_embeddings, attention_mask, metadata_emb=None):
        B, N, _ = visual_embeddings.shape
        pos_idx = torch.arange(N, device=visual_embeddings.device).clamp(0, self.max_views - 1)
        tokens = visual_embeddings + self.view_pos(pos_idx).unsqueeze(0)
        if metadata_emb is not None:
            meta_token = self.meta_proj(metadata_emb).unsqueeze(1)
            meta_token = meta_token + self.view_pos(torch.tensor(self.max_views, device=tokens.device)).unsqueeze(0)
            tokens = torch.cat([meta_token, tokens], dim=1)
            meta_pad = torch.zeros(B, 1, dtype=torch.bool, device=tokens.device)
            key_padding_mask = torch.cat([meta_pad, ~attention_mask], dim=1)
        else:
            key_padding_mask = ~attention_mask
        attn_out, _ = self.self_attn(tokens, tokens, tokens, key_padding_mask=key_padding_mask)
        tokens = self.norm1(tokens + attn_out)
        tokens = self.norm2(tokens + self.ffn(tokens))
        if metadata_emb is not None:
            valid_mask = torch.cat([
                torch.ones(B, 1, dtype=torch.bool, device=tokens.device),
                attention_mask
            ], dim=1)
        else:
            valid_mask = attention_mask
        pooled = (tokens * valid_mask.unsqueeze(-1).float()).sum(dim=1) / \
                 valid_mask.sum(dim=1, keepdim=True).float().clamp(min=1)
        if metadata_emb is not None:
            out = torch.cat([pooled, metadata_emb], dim=-1)
        else:
            zero_meta = torch.zeros(B, self.metadata_dim, device=pooled.device)
            out = torch.cat([pooled, zero_meta], dim=-1)
        return out


log("AttentionFusion defined.")

In [ ]:
# ═══ CELL 14: ArcFace Head + CenterLoss + TemperatureScaler ═══
class ArcFaceHead(nn.Module):
    def __init__(self, in_features, num_classes, s=30.0, m=0.50):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.s = s
        self.m = m
        self.num_classes = num_classes

    def forward(self, embeddings, labels=None):
        W = F.normalize(self.weight, dim=1)
        E = F.normalize(embeddings, dim=1)
        cosine = E @ W.T
        if labels is not None:
            theta = torch.acos(cosine.clamp(-1 + 1e-7, 1 - 1e-7))
            target_logits = torch.cos(theta + self.m)
            one_hot = F.one_hot(labels, self.num_classes).float()
            cosine = one_hot * target_logits + (1 - one_hot) * cosine
        return cosine * self.s


class CenterLoss(nn.Module):
    def __init__(self, num_classes, feat_dim):
        super().__init__()
        self.centers = nn.Parameter(torch.randn(num_classes, feat_dim))
    def forward(self, x, labels):
        batch_centers = self.centers[labels]
        return ((x - batch_centers) ** 2).sum(dim=1).mean()


class TemperatureScaler(nn.Module):
    def __init__(self, num_combos=16):
        super().__init__()
        self.log_temp = nn.Parameter(torch.zeros(num_combos))
    def forward(self, logits, combo_idx=0):
        temp = self.log_temp[combo_idx].exp()
        return logits / temp


log("ArcFaceHead, CenterLoss, TemperatureScaler defined.")

In [ ]:
# ═══ CELL 15: Full Multi-View Model ═══
class MultiViewModel(nn.Module):
    def __init__(self, backbone_name='convnextv2_tiny.fcmae_ft_in22k_in1k',
                 d_model=512, metadata_dim=64, num_classes=1000, lora_rank=16):
        super().__init__()
        self.backbone = ViewConditionedBackbone(backbone_name, d_model, lora_rank)
        self.metadata_encoder = MetadataEncoder(out_dim=metadata_dim)
        self.fusion = AttentionFusion(d_model, metadata_dim)
        feat_dim = self.fusion.output_dim
        self.arcface = ArcFaceHead(feat_dim, num_classes)
        self.center_loss = CenterLoss(num_classes, feat_dim)

    def forward(self, images, view_idx, attention_mask, metadata_indices, labels=None):
        visual_emb = self.backbone(images, view_idx, attention_mask)
        meta_embeds = []
        for name in ['habitat', 'substrate', 'smell', 'country']:
            idx = metadata_indices.get(name, torch.zeros(images.size(0), dtype=torch.long, device=images.device))
            meta_embeds.append(self.metadata_encoder.embeddings[name](idx))
        meta_concat = torch.cat(meta_embeds, dim=-1)
        metadata_emb = self.metadata_encoder.mlp(meta_concat)
        obs_emb = self.fusion(visual_emb, attention_mask, metadata_emb)
        logits = self.arcface(obs_emb, labels)
        return logits, obs_emb


log("MultiViewModel defined.")

In [ ]:
# ═══ CELL 16: Metadata vocab + model smoke test ═══
def build_metadata_vocab(observations):
    fields = ('habitat', 'substrate', 'smell', 'country')
    vocab = {f: {'<unk>': 0, 'unknown': 1} for f in fields}
    for obs in observations:
        for f in fields:
            val = obs.get(f, 'unknown')
            if val and val not in vocab[f]:
                vocab[f][val] = len(vocab[f])
    return vocab


metadata_vocab = build_metadata_vocab(train_obs + val_obs)
METADATA_VOCAB_SIZES = {f: len(v) for f, v in metadata_vocab.items()}
log(f"Metadata vocab sizes: {METADATA_VOCAB_SIZES}")


def encode_metadata_batch(metadata_raw):
    out = {}
    for field_name in ['habitat', 'substrate', 'smell', 'country']:
        vals = metadata_raw.get(field_name, [])
        idxs = [metadata_vocab[field_name].get(v, 0) for v in vals]
        out[field_name] = torch.tensor(idxs, dtype=torch.long, device=DEVICE)
    return out


log("Testing model forward pass...")
test_model = MultiViewModel(
    num_classes=min(NUM_CLASSES, 100),
    d_model=256, metadata_dim=32,
).to(DEVICE)

test_images = torch.randn(4, 4, 3, 224, 224).to(DEVICE)
test_view_idx = torch.tensor([[0,1,2,3],[0,1,2,3],[0,1,0,0],[0,1,2,3]]).to(DEVICE)
test_mask = torch.tensor([[True,True,True,True],[True,True,True,True],
                           [True,True,False,False],[True,True,True,False]]).to(DEVICE)
test_meta = {k: torch.tensor([0,0,0,0]).to(DEVICE) for k in ['habitat','substrate','smell','country']}
test_labels = torch.tensor([0,1,2,3]).to(DEVICE)

test_logits, test_emb = test_model(test_images, test_view_idx, test_mask, test_meta, test_labels)
param_count = sum(p.numel() for p in test_model.parameters()) / 1e6
log(f"  Forward OK | logits: {test_logits.shape} | emb: {test_emb.shape} | params: {param_count:.1f}M")
del test_model
torch.cuda.empty_cache()

# ═══ E20b_RESUME probe (load happens AFTER model build in next cells) ═══
_WEIGHT_CANDIDATES = [
    Path('/kaggle/input/visionsetil-e20-weights/best.pt'),
    Path('/kaggle/input/visionsetil-e20-weights/models/best.pt'),
    Path('/kaggle/input/datasets/alonsoalviraaaa/visionsetil-e20-weights/best.pt'),
    Path('/kaggle/input/datasets/alonsoalviraaaa/visionsetil-e20-weights/models/best.pt'),
    Path('/kaggle/input/alonsoalviraaaa/visionsetil-e20-weights/best.pt'),
    Path('/kaggle/working/models/best.pt'),
]
_E20B_WEIGHT_PATH = next((p for p in _WEIGHT_CANDIDATES if p.is_file()), None)
if _E20B_WEIGHT_PATH is not None:
    log(f"E20b FT: will load weights from {_E20B_WEIGHT_PATH} after model build")
else:
    log("E20b FT WARNING: no pretrained best.pt found yet — check visionsetil-e20-weights dataset mount")


In [ ]:
# ═══ CELL 17: Train Config + Deadly Species (DO3) ═══
DEADLY_SPECIES = {
    'amanita phalloides', 'amanita virosa', 'amanita bisporigera',
    'amanita ocreata', 'amanita smithiana', 'amanita proxima',
    'amanita exitialis', 'amanita magnivelaris',
    'amanita suballiacea', 'amanita tenuifolia', 'amanita verna',
    'galerina marginata', 'galerina autumnalis', 'galerina venenata',
    'lepiota castanea', 'lepiota helveola', 'lepiota subincarnata',
    'lepiota brunneoincarnata', 'lepiota josserandii',
    'cortinarius orellanus', 'cortinarius rubellus', 'cortinarius speciosissimus',
    'podostroma cornu-damae', 'funoria fascicularis',
    'naematoloma fasciculare', 'hypholoma fasciculare',
}

deadly_label_indices = set()
for sp, idx in label2idx.items():
    if sp.lower() in DEADLY_SPECIES or sp.lower() in DEADLY_FORCE:
        deadly_label_indices.add(idx)
log(f"Deadly species in dataset: {len(deadly_label_indices)}")

class_weights = torch.ones(NUM_CLASSES, device=DEVICE)
for di in deadly_label_indices:
    if 0 <= di < NUM_CLASSES:
        class_weights[di] = 12.0
log(f"E20 deadly class_weights x12 n={len(deadly_label_indices)}")
# Loop-ML iters 26–31: hard-neg Lepiota (subincarnata↔cristata) + castanea.
# Confuser cristata is NOT deadly but must appear more often in batches.
_HARD_NEG = {
    'lepiota subincarnata': 28.0,  # E20b FT boost (deadly)
    'lepiota castanea': 22.0,      # deadly
    'lepiota cristata': 16.0,      # confuser hard-neg (not deadly)
    'amanita phalloides': 14.0,
    'amanita citrina': 8.0,
}
_hn_n = 0
for _sp, _idx in label2idx.items():
    _k = str(_sp).strip().lower()
    if _k in _HARD_NEG and 0 <= int(_idx) < NUM_CLASSES:
        class_weights[int(_idx)] = max(float(class_weights[int(_idx)]), float(_HARD_NEG[_k]))
        _hn_n += 1
log(f"E20 hard-neg class_weights n={_hn_n} (loop-ML; orientation only, product_unlock=false)")



@dataclass
class TrainConfig:
    backbone: str = 'convnextv2_tiny.fcmae_ft_in22k_in1k'
    d_model: int = 512
    metadata_dim: int = 64
    lora_rank: int = 16
    epochs: int = 12  # E20b Lepiota FT
    patience: int = 10  # E20 dual early-stop
    warmup_epochs: int = 2  # E20
    swa_start_epoch: int = 8  # E20b FT
    batch_size: int = 10  # E20 multi-view base (scaled x N_GPU)
    lr_head: float = 3e-4
    lr_backbone: float = 2e-5
    weight_decay: float = 0.01
    label_smoothing: float = 0.1
    use_swa: bool = True
    center_loss_weight: float = 0.05
    max_grad_norm: float = 1.0
    amp: bool = True
    mixup_alpha: float = 0.2
    seed: int = 42


cfg = TrainConfig()

# T4x2: scale batch when 2 GPUs available (DataParallel splits batch)
if globals().get('N_GPU', 0) >= 2:
    cfg.batch_size = min(cfg.batch_size * 2, 20)
    log(f'E20 multi-GPU: N_GPU={N_GPU}, batch_size={cfg.batch_size}')

if len(train_obs) < 100:
    cfg.epochs = 12  # E20b FT fixed
    cfg.batch_size = 4
    cfg.d_model = 128
    cfg.metadata_dim = 16
    log(f"WARNING: Small dataset ({len(train_obs)} obs). Smoke-test config")

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
torch.cuda.manual_seed_all(cfg.seed)

log(f"Config: backbone={cfg.backbone}, epochs={cfg.epochs}, batch={cfg.batch_size}")


In [ ]:
# ═══ CELL 18: Build full model + optimizer + output dirs ═══
model = MultiViewModel(
    backbone_name=cfg.backbone,
    d_model=cfg.d_model,
    metadata_dim=cfg.metadata_dim,
    num_classes=NUM_CLASSES,
    lora_rank=cfg.lora_rank,
).to(DEVICE)

# T4x2: DataParallel when 2+ GPUs; graceful single-GPU fallback
if globals().get('N_GPU', 0) >= 2:
    model = nn.DataParallel(model)
    log(f"E20 DataParallel enabled on {N_GPU} GPUs")
else:
    log(f"E20 single-device training (N_GPU={globals().get('N_GPU', 0)})")

def _unwrap(m):
    return m.module if isinstance(m, nn.DataParallel) else m

def _model_state(m):
    # Always unwrapped state_dict (never use bare dir() inside nested fns)
    return _unwrap(m).state_dict()

def _load_model_state(m, state_dict):
    # Load unwrapped or DP-prefixed state into current model layout
    try:
        m.load_state_dict(state_dict)
        return
    except RuntimeError:
        pass
    # strip or add module. prefix for T4x2 <-> single GPU resume
    sd = state_dict
    if any(k.startswith('module.') for k in sd.keys()):
        sd = {k[7:] if k.startswith('module.') else k: v for k, v in sd.items()}
        try:
            _unwrap(m).load_state_dict(sd)
            return
        except RuntimeError:
            pass
    else:
        try:
            _unwrap(m).load_state_dict(sd)
            return
        except RuntimeError:
            pass
        try:
            m.load_state_dict({'module.' + k: v for k, v in sd.items()})
            return
        except RuntimeError:
            pass
    raise RuntimeError('Failed to load model_state (DataParallel key mismatch)')

param_count = sum(p.numel() for p in _unwrap(model).parameters()) / 1e6
log(f"Model parameters: {param_count:.1f}M")

_m = _unwrap(model)
backbone_params = list(_m.backbone.backbone.parameters())
head_params = [p for n, p in _m.named_parameters() if not n.startswith('backbone.backbone.')]
optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': cfg.lr_backbone},
    {'params': head_params, 'lr': cfg.lr_head},
], weight_decay=cfg.weight_decay)

scaler = torch.amp.GradScaler('cuda', enabled=cfg.amp)

swa_model = None
if cfg.use_swa:
    swa_model = torch.optim.swa_utils.AveragedModel(model)

OUT_DIR = Path('/kaggle/working/models')
OUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUT_DIR / 'checkpoint_latest.pt'

log("Optimizer + AMP + SWA ready.")

# ═══ E20b_RESUME: load MAP@3 weights for Lepiota fine-tune ═══
_WEIGHT_CANDIDATES = [
    # Kaggle CLI mount layouts (slug-only and owner/slug)
    Path('/kaggle/input/visionsetil-e20-weights/best.pt'),
    Path('/kaggle/input/visionsetil-e20-weights/models/best.pt'),
    Path('/kaggle/input/datasets/alonsoalviraaaa/visionsetil-e20-weights/best.pt'),
    Path('/kaggle/input/datasets/alonsoalviraaaa/visionsetil-e20-weights/models/best.pt'),
    Path('/kaggle/input/alonsoalviraaaa/visionsetil-e20-weights/best.pt'),
    Path('/kaggle/working/models/best.pt'),
]
_loaded_ft = False
_cand_list = list(_WEIGHT_CANDIDATES)
if globals().get("_E20B_WEIGHT_PATH") is not None:
    _cand_list = [globals()["_E20B_WEIGHT_PATH"]] + [
        p for p in _cand_list if p != globals()["_E20B_WEIGHT_PATH"]
    ]
for _wp in _cand_list:
    if _wp.is_file():
        try:
            _ckpt = torch.load(_wp, map_location=DEVICE, weights_only=False)
            _sd = _ckpt.get('model_state') or _ckpt.get('state_dict') or _ckpt
            try:
                model.load_state_dict(_sd, strict=False)
            except Exception:
                # DataParallel module. prefix
                if any(k.startswith('module.') for k in _sd.keys()):
                    model.load_state_dict(_sd, strict=False)
                else:
                    model.load_state_dict({'module.' + k: v for k, v in _sd.items()}, strict=False)
            log(f"E20b FT: loaded weights from {_wp} (strict=False)")
            _loaded_ft = True
            break
        except Exception as _e:
            log(f"E20b FT: failed load {_wp}: {_e}")
if not _loaded_ft:
    log("E20b FT WARNING: no pretrained best.pt found — training from current init")


In [ ]:
# ═══ CELL 19: Training Loop ═══
def map_at_3(probs, labels):
    top3 = np.argsort(-probs, axis=1)[:, :3]
    score = 0.0
    for i, label in enumerate(labels):
        if label in top3[i]:
            rank = list(top3[i]).index(label)
            score += 1.0 / (rank + 1)
    return score / max(len(labels), 1)


def save_checkpoint(epoch, model, optimizer, best_map3, best_epoch, history):
    torch.save({
        'epoch': epoch,
        'model_state': _model_state(model),
        'optimizer_state': optimizer.state_dict(),
        'best_map3': best_map3,
        'best_epoch': best_epoch,
        'history': history,
    }, CHECKPOINT_PATH)


def load_checkpoint_if_exists():
    if CHECKPOINT_PATH.exists():
        ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
        log(f"Resuming from checkpoint: epoch={ckpt['epoch']}")
        return ckpt
    return None


def train_one_epoch(model, loader, optimizer, epoch, image_size):
    model.train()
    total_loss = 0.0
    n_obs = 0
    epoch_start = time.time()

    for batch_idx, batch in enumerate(loader):
        images = batch['images'].to(DEVICE)
        view_idx = batch['view_idx'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        meta = encode_metadata_batch(batch['metadata_raw'])

        use_mixup = cfg.mixup_alpha > 0 and random.random() < 0.5

        with torch.amp.autocast('cuda', enabled=cfg.amp):
            logits, emb = model(
                images, view_idx, attention_mask, meta,
                labels=labels if not use_mixup else None
            )
            if use_mixup:
                soft = torch.full_like(logits, cfg.label_smoothing / max(NUM_CLASSES - 1, 1))
                soft[range(len(labels)), labels] = 1.0 - cfg.label_smoothing
                loss_cls = -(soft * F.log_softmax(logits, dim=-1)).sum(-1).mean()
            else:
                loss_cls = F.cross_entropy(logits, labels, weight=class_weights, label_smoothing=cfg.label_smoothing)
            # E20: push deadly true class into top-3
            if len(deadly_label_indices) > 0:
                _didx = deadly_label_indices
                _is_d = torch.tensor([int(l) in _didx for l in labels.tolist()], device=logits.device)
                if _is_d.any():
                    _true = logits.gather(1, labels.unsqueeze(1)).squeeze(1)
                    _kth = logits.topk(3, dim=-1).values[:, -1]
                    loss_cls = loss_cls + 0.75 * torch.relu(_kth - _true + 0.1)[_is_d].mean()
            loss = loss_cls
            if cfg.center_loss_weight > 0:
                cl = model.center_loss(emb, labels)
                loss = loss + cfg.center_loss_weight * cl

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * len(labels)
        n_obs += len(labels)

        if batch_idx % 10 == 0:
            elapsed = time.time() - epoch_start
            batches_done = batch_idx + 1
            batches_total = len(loader)
            eta_sec = (elapsed / batches_done) * (batches_total - batches_done)
            log(f"  Ep{epoch} B{batch_idx}/{batches_total} | loss={loss.item():.4f} | "
                f"{elapsed:.0f}s | ETA {eta_sec/60:.1f}min")

    avg_loss = total_loss / max(n_obs, 1)
    epoch_time = time.time() - epoch_start
    log(f"  Ep{epoch} DONE | avg_loss={avg_loss:.4f} | time={epoch_time:.0f}s ({epoch_time/60:.1f}min)")
    return avg_loss


@torch.no_grad()
def validate(model, loader, image_size):
    model.eval()
    all_probs, all_labels = [], []
    for batch in loader:
        images = batch['images'].to(DEVICE)
        view_idx = batch['view_idx'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        meta = encode_metadata_batch(batch['metadata_raw'])
        logits, _ = model(images, view_idx, attention_mask, meta)
        probs = F.softmax(logits, dim=-1)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    preds = all_probs.argmax(axis=1)
    acc = (preds == all_labels).mean()
    map3 = map_at_3(all_probs, all_labels)
    f1 = f1_score(all_labels, preds, average='macro', zero_division=0)
    deadly_rec = 0.0
    if len(deadly_label_indices) > 0:
        top3 = np.argsort(-all_probs, axis=1)[:, :3]
        dmask = np.array([int(l) in deadly_label_indices for l in all_labels])
        if dmask.any():
            hits = 0
            for i, lab in enumerate(all_labels):
                if dmask[i] and lab in top3[i]:
                    hits += 1
            deadly_rec = hits / max(int(dmask.sum()), 1)
    return {'acc': acc, 'map3': map3, 'f1': f1, 'deadly3': deadly_rec}


best_map3 = 0.0
best_deadly = 0.0
best_epoch = -1
history = []
epochs_no_improve = 0

ckpt = load_checkpoint_if_exists()
start_epoch = 0
if ckpt is not None:
    _load_model_state(model, ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    start_epoch = ckpt['epoch'] + 1
    best_map3 = ckpt.get('best_map3', 0.0)
    best_epoch = ckpt.get('best_epoch', -1)
    history = ckpt.get('history', [])

_prev_img_size = None
train_loader = None
val_loader = None

for epoch in range(start_epoch, cfg.epochs):
    # E20: fixed 224 (T4-safe multi-view)
    img_size = 224

    if epoch < cfg.warmup_epochs:
        for p in _unwrap(model).backbone.backbone.parameters():
            p.requires_grad = False
        log(f"Ep{epoch}: Backbone FROZEN (warmup)")
    else:
        for p in _unwrap(model).backbone.backbone.parameters():
            p.requires_grad = True

    if img_size != _prev_img_size:
        train_ds = MultiViewDataset(train_obs, label2idx, image_size=img_size, augment=True)
        val_ds = MultiViewDataset(val_obs, label2idx, image_size=img_size, augment=False)
        train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                                  collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)
        val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                                collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)
        _prev_img_size = img_size
        log(f"  Loaders: {len(train_loader)} train batches, {len(val_loader)} val batches")

    log(f"{'='*60}")
    log(f"EPOCH {epoch}/{cfg.epochs - 1}")
    log(f"{'='*60}")

    train_loss = train_one_epoch(model, train_loader, optimizer, epoch, img_size)
    val_metrics = validate(model, val_loader, img_size)

    history.append({
        'epoch': epoch, 'train_loss': train_loss,
        'val_acc': val_metrics['acc'], 'val_map3': val_metrics['map3'],
        'val_f1': val_metrics['f1'],
        'val_deadly3': val_metrics.get('deadly3', 0.0),
    })

    log(f"Ep{epoch} RESULT | loss={train_loss:.4f} | acc={val_metrics['acc']:.4f} | "
        f"map3={val_metrics['map3']:.4f} | f1={val_metrics['f1']:.4f} | "
        f"deadly3={val_metrics.get('deadly3', 0):.4f}")

    # Dual early-stop: patience resets on MAP@3 OR deadly@3 improvement.
    # best.pt = MAP@3 weights (primary product metric load); best_deadly.pt = deadly@3 peak.
    improved = False
    if val_metrics['map3'] > best_map3:
        best_map3 = val_metrics['map3']
        best_epoch = epoch
        improved = True
        torch.save({
            'epoch': epoch,
            'model_state': _model_state(model),
            'config': {'d_model': cfg.d_model, 'metadata_dim': cfg.metadata_dim,
                       'num_classes': NUM_CLASSES, 'lora_rank': cfg.lora_rank},
            'label2idx': label2idx,
            'metadata_vocab': metadata_vocab,
            'val_map3': float(best_map3),
            'val_deadly3': float(val_metrics.get('deadly3', 0.0) or 0.0),
            'checkpoint_kind': 'best_map3',
        }, OUT_DIR / 'best.pt')
        log(f"  ★ New best MAP@3: {best_map3:.4f} — saved best.pt!")
    d3 = float(val_metrics.get('deadly3', 0.0) or 0.0)
    if d3 > best_deadly + 1e-6:
        best_deadly = d3
        improved = True
        torch.save({
            'epoch': epoch,
            'model_state': _model_state(model),
            'config': {'d_model': cfg.d_model, 'metadata_dim': cfg.metadata_dim,
                       'num_classes': NUM_CLASSES, 'lora_rank': cfg.lora_rank},
            'label2idx': label2idx,
            'metadata_vocab': metadata_vocab,
            'val_map3': float(val_metrics['map3']),
            'val_deadly3': float(best_deadly),
            'checkpoint_kind': 'best_deadly3',
        }, OUT_DIR / 'best_deadly.pt')
        log(f"  ★ New best val deadly@3: {best_deadly:.4f} — saved best_deadly.pt!")
    if improved:
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        log(f"  No improvement for {epochs_no_improve} epoch(s).")

    save_checkpoint(epoch, model, optimizer, best_map3, best_epoch, history)

    if swa_model is not None and epoch >= cfg.swa_start_epoch:
        swa_model.update_parameters(model)

    if epochs_no_improve >= cfg.patience:
        log(f"⚠️ Early stopping!")
        break

log(f"\nTRAINING COMPLETE! Best MAP@3: {best_map3:.4f} @ epoch {best_epoch}")

In [ ]:
# ═══ CELL 20: SWA finalize + Temperature calibration ═══
if swa_model is not None and best_epoch >= cfg.swa_start_epoch:
    log("Updating SWA BatchNorm...")
    torch.optim.swa_utils.update_bn(train_loader, swa_model, device=DEVICE)
    torch.save({
        'model_state': swa_model.state_dict(),
        'config': {'d_model': cfg.d_model, 'metadata_dim': cfg.metadata_dim, 'num_classes': NUM_CLASSES},
        'label2idx': label2idx,
    }, OUT_DIR / 'swa.pt')
    log("SWA model saved.")

log("Calibrating temperature...")
temp_scaler = TemperatureScaler(num_combos=16).to(DEVICE)
temp_opt = torch.optim.LBFGS([temp_scaler.log_temp], lr=0.01, max_iter=50)

logits_list, labels_list = [], []
model.eval()
with torch.no_grad():
    for batch in val_loader:
        images = batch['images'].to(DEVICE)
        view_idx = batch['view_idx'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        meta = encode_metadata_batch(batch['metadata_raw'])
        logits, _ = model(images, view_idx, attention_mask, meta)
        logits_list.append(logits)
        labels_list.append(labels)

if logits_list:
    all_logits = torch.cat(logits_list)
    all_labels_t = torch.cat(labels_list)
    def closure():
        temp_opt.zero_grad()
        scaled = temp_scaler(all_logits, combo_idx=0)
        loss = F.cross_entropy(scaled, all_labels_t)
        loss.backward()
        return loss
    temp_opt.step(closure)
    learned_temp = temp_scaler.log_temp[0].exp().item()
    log(f"Learned temperature: {learned_temp:.4f}")
    torch.save(temp_scaler.state_dict(), OUT_DIR / 'temperature_scaler.pt')
else:
    learned_temp = 1.5
    log("WARNING: No validation logits. Using default T=1.5")

In [ ]:
# ═══ CELL 21: Final test evaluation + safety + per-species (DO3, DO10) ═══
log("=" * 60)
log("FINAL TEST EVALUATION")
log("=" * 60)

best_ckpt = torch.load(OUT_DIR / 'best.pt', map_location=DEVICE, weights_only=False)
_load_model_state(model, best_ckpt['model_state'])
model.eval()
log('Loaded best.pt (MAP@3 checkpoint); best_deadly.pt saved when deadly@3 improved')

test_ds = MultiViewDataset(test_obs, label2idx, image_size=224, augment=False)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False,
                         collate_fn=collate_fn, num_workers=NUM_WORKERS)

all_probs, all_labels, all_preds, all_logits_list = [], [], [], []
with torch.no_grad():
    for batch in test_loader:
        images = batch['images'].to(DEVICE)
        view_idx = batch['view_idx'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        meta = encode_metadata_batch(batch['metadata_raw'])
        logits, _ = model(images, view_idx, attention_mask, meta)
        # Loop-ML: raw pre-temperature logits for ECE re-T (lab only)
        all_logits_list.append(logits.float().cpu().numpy())
        scaled = temp_scaler(logits, combo_idx=0)
        probs = F.softmax(scaled, dim=-1)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
        all_preds.append(probs.argmax(dim=-1).cpu().numpy())

all_probs = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)
all_preds = np.concatenate(all_preds)
all_logits = np.concatenate(all_logits_list)

test_acc = (all_preds == all_labels).mean()
test_map3 = map_at_3(all_probs, all_labels)
test_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
test_bal = balanced_accuracy_score(all_labels, all_preds)
max_probs = all_probs.max(axis=1)
# Dual ECE honesty: primary = train-published T (temp_scaler fit on val, applied here).
# mean|max_p-correct| is a crude proxy (not binned ECE); still labeled train_published.
# Posthoc T-grid on test logits is lab-only suite work — never sold as primary / never unlocks.
ece = np.mean(np.abs(max_probs - (all_preds == all_labels).astype(float)))
test_ece_train_published = float(ece)
ece_primary_label = 'train_published'
claim_train_published = True
posthoc_separate = True
test_ece_posthoc = None  # lab suite may recompute; not primary
temperature_train = float(learned_temp)
temperature_posthoc = None

log(f"  Accuracy:       {test_acc:.4f}")
log(f"  MAP@3:          {test_map3:.4f}")
log(f"  Macro-F1:       {test_f1:.4f}")
log(f"  Balanced Acc:   {test_bal:.4f}")
log(f"  ECE primary (train_published): {test_ece_train_published:.4f}  [proxy; not posthoc]")
log("  product_unlock=false (orientation only)")

# DO3: Safety Recall Deadly = TRUE @3 (E19 mislabeled top-1 as safety_recall_deadly)
# FAIL-CLOSED: n_deadly==0 must NOT green-pass expand gate (no vacuous 1.0)
deadly_mask = np.array([l in deadly_label_indices for l in all_labels])
n_deadly = int(deadly_mask.sum())
if n_deadly > 0:
    # @1 diagnostic
    deadly_correct_1 = (all_preds[deadly_mask] == all_labels[deadly_mask]).sum()
    safety_recall_deadly_at_1 = float(deadly_correct_1 / n_deadly)
    # @3 product gate (true class in top-3 among deadly-labeled samples)
    top3 = np.argsort(-all_probs, axis=1)[:, :3]
    hits3 = 0
    for i, lab in enumerate(all_labels):
        if deadly_mask[i] and lab in top3[i]:
            hits3 += 1
    safety_recall_deadly_at_3 = float(hits3 / n_deadly)
    # Primary field name = @3 (gates use this)
    safety_recall_deadly = safety_recall_deadly_at_3
    deadly_gate_status = 'ok'
    log(f"  🔴 DEADLY species in test (GBIF pure): {n_deadly}")
    log(f"  🔴 safety_recall_deadly_at_1: {safety_recall_deadly_at_1:.4f}")
    log(f"  🔴 safety_recall_deadly_at_3 (=safety_recall_deadly): {safety_recall_deadly_at_3:.4f}")
else:
    # Fail-closed: unevaluable — numeric 0.0 so threshold checks cannot pass
    safety_recall_deadly = 0.0
    safety_recall_deadly_at_1 = 0.0
    safety_recall_deadly_at_3 = 0.0
    deadly_gate_status = 'unevaluable'
    log('  FATAL-soft: 0 deadly samples in pure GBIF test — deadly gate UNEVALUABLE')
    log('  safety_recall_deadly set to 0.0 (fail-closed; will not pass expand gate)')

# IC 95%
log("Computing IC 95%...")
n_bootstrap = 1000
map3_scores = []
n = len(all_labels)
for _ in range(n_bootstrap):
    idx = np.random.choice(n, n, replace=True)
    map3_scores.append(map_at_3(all_probs[idx], all_labels[idx]))
ci_low = np.percentile(map3_scores, 2.5)
ci_high = np.percentile(map3_scores, 97.5)
log(f"  MAP@3 95% CI:   [{ci_low:.4f}, {ci_high:.4f}]")

# Per-species diagnostics
log("\nPer-species accuracy (worst 20):")
per_species = defaultdict(list)
for pred, label in zip(all_preds, all_labels):
    per_species[label].append(pred == label)
worst = sorted(per_species.items(), key=lambda x: np.mean(x[1]))[:20]
for species_idx, correct_list in worst:
    species_name = idx2label.get(species_idx, f"class_{species_idx}")
    acc = np.mean(correct_list)
    is_deadly = "💀" if species_idx in deadly_label_indices else "  "
    log(f"  {is_deadly} {species_name[:40]:40s}: {acc:.2f}")


In [ ]:
# ═══ CELL 22: Export all artifacts (DO8) ═══
final_metrics = {
    'test_accuracy': float(test_acc),
    'test_map_at_3': float(test_map3),
    'test_map_at_3_ci_low': float(ci_low),
    'test_map_at_3_ci_high': float(ci_high),
    'test_f1_macro': float(test_f1),
    'test_balanced_accuracy': float(test_bal),
    'test_ece': float(test_ece_train_published),  # alias of train-published primary
    'test_ece_train_published': float(test_ece_train_published),
    'ece_primary': ece_primary_label,
    'ece_primary_is_train_published': True,
    'claim_train_published': bool(claim_train_published),
    'posthoc_separate': True,
    'test_ece_posthoc': test_ece_posthoc,  # lab-only sidecar; never primary; None until suite
    'temperature': float(temperature_train),
    'temperature_train': float(temperature_train),
    'temperature_posthoc': temperature_posthoc,
    'product_unlock': False,
    'can_auto_unlock': False,
    'forage_permission': False,
    'consumption_permission': False,
    'lab_only': True,
    'ece_note': (
        'Primary ECE is train-published (val-calibrated T). '
        'Posthoc is separate lab-only; never sell as primary; never product_unlock from ECE. '
        'Value is mean|max_p-correct| proxy unless suite recomputes binned ECE.'
    ),
    'safety_recall_deadly': float(safety_recall_deadly),  # @3 (product gate); 0.0 if n_deadly==0 fail-closed
    'safety_recall_deadly_at_1': float(safety_recall_deadly_at_1),
    'safety_recall_deadly_at_3': float(safety_recall_deadly_at_3),
    'safety_recall_deadly_definition': 'top-3 among deadly-labeled samples (true class in top-3)',
    'n_deadly_in_test': int(n_deadly),
    'deadly_gate_status': str(deadly_gate_status),
    'deadly_gate_pass_expand': bool(n_deadly > 0 and safety_recall_deadly >= 0.50),
    'deadly_gate_pass_soft': bool(n_deadly > 0 and safety_recall_deadly >= 0.90),
    'eval_protocol': 'source_holdout_e20b_lepiota_ft',
    'test_domain': 'gbif_es_only',
    'train_domain': 'fungitastic_plus_soft_non_gbif',
    'primary_checkpoint': 'best.pt (MAP@3); best_deadly.pt also saved on deadly@3 peak',
    'split_artifacts': ['train_obs.json', 'val_obs.json', 'test_obs.json', 'split_manifest.json'],
    'best_val_map3': float(best_map3),
    'best_epoch': int(best_epoch),
    'num_classes': int(NUM_CLASSES),
    'num_train_obs': int(len(train_obs)),
    'num_val_obs': int(len(val_obs)),
    'num_test_obs': int(len(test_obs)),
    # temperature_train exported above (dual ECE honesty)
    'model_params_M': float(param_count),
    'databases_used': list(globals().get('DATABASES_USED_EFFECTIVE') or (
        sorted(df['source_db'].unique().tolist()) if len(df) and 'source_db' in df.columns else [])),
    'datasets_mounted': list(ALL_DATASETS.keys()),
    'subsample_config': {
        'max_species': 40, 'max_obs': 200, 'max_obs_deadly': 400,
        'experiment': 'E20b-lepiota-ft', 'allowlist': 'industrial_v1',
        'epochs': 12,
        'protocol': 'train=FT(+soft non-GBIF); val=FT holdout; test=GBIF ES pure',
        'sources_train': ['fungitastic'],
        'sources_test': ['gbif_es'],
        'sources_optional_train': ['mush215'],
        'near_dup': True,
        'persist_split_artifacts': True,
        'gate_map': 0.22, 'gate_deadly_at_3': 0.50,
        'soft_gate_map': 0.25, 'soft_gate_deadly_at_3': 0.90,
        'prefer_cc_ok': True,
        'multi_gpu': 'DataParallel if N_GPU>=2 (T4x2)',
        'note': 'honest gates lower than E19 mixed; orientation only; no expand to 80 until pass',
    },
    'deadly_species_known': len(DEADLY_SPECIES),
    'deadly_species_in_dataset': len(deadly_label_indices),
    'version': 'v20b-E20-lepiota-ft',
    'n_gpu': int(globals().get('N_GPU', 0)),
    'attribution': 'FungiTastic (Picek et al.) train + GBIF ES StillImage pure test; educational orientation only',
}

with open(OUT_DIR / 'metrics.json', 'w') as f:
    json.dump(final_metrics, f, indent=2)
with open(OUT_DIR / 'label2idx.json', 'w') as f:
    json.dump(label2idx, f, indent=2)
with open(OUT_DIR / 'training_history.json', 'w') as f:
    json.dump(history, f, indent=2)
# Loop-ML: logits = pre-temperature model outputs; probs = post-T softmax.
np.savez(
    OUT_DIR / 'test_predictions.npz',
    probs=all_probs,
    preds=all_preds,
    labels=all_labels,
    logits=all_logits,
)

log("Artifacts saved:")
for f in sorted(OUT_DIR.iterdir()):
    size = f.stat().st_size
    size_str = f"{size/1e6:.1f} MB" if size > 1e6 else f"{size/1e3:.1f} KB"
    log(f"  {f.name}: {size_str}")

log(f"\n{'='*60}")
log(f"TRAINING COMPLETE! (v20b-E20-lepiota-ft)")
log(f"  MAP@3:          {test_map3:.4f} (CI: [{ci_low:.4f}, {ci_high:.4f}])")
log(f"  Accuracy:       {test_acc:.4f}")
log(f"  Macro-F1:       {test_f1:.4f}")
log(f"  ECE primary (train_published): {test_ece_train_published:.4f}")
log(f"  Safety Recall:  {safety_recall_deadly:.4f}")
log(f"  DBs used:       {final_metrics.get('databases_used', [])}")
log(f"  Protocol:       source_holdout test=GBIF pure")
log(f"  N_GPU:          {globals().get('N_GPU', 0)}")
log(f"{'='*60}")

log("\n📋 E20 HONEST GATES (source hold-out):")
log(f"  DO1: Runs < 8h ............... ✅")
log(f"  DO2: expand-to-80 MAP@3≥0.22 . {'✅' if test_map3 >= 0.22 else '⚠️'} ({test_map3:.4f})")
log(f"  DO2b: soft-gate A MAP@3≥0.25 . {'✅' if test_map3 >= 0.25 else '⚠️'} ({test_map3:.4f})")
_dg = (n_deadly > 0 and safety_recall_deadly >= 0.50)
_dgs = (n_deadly > 0 and safety_recall_deadly >= 0.90)
log(f"  DO3: expand deadly@3≥0.50 .... {'✅' if _dg else '❌'} (val={safety_recall_deadly:.4f} n_deadly={n_deadly} status={deadly_gate_status})")
log(f"  DO3b: soft-gate deadly@3≥0.90 {'✅' if _dgs else '❌'} (val={safety_recall_deadly:.4f} n_deadly={n_deadly})")
log(f"  DO3c: deadly@1 (diagnostic) .. {safety_recall_deadly_at_1:.4f}")
log(f"  DO3d: fail-closed n_deadly>0 . {'✅' if n_deadly > 0 else '❌ UNEVALUABLE'}")
log(f"  DO4: Logging real-time ....... ✅")
log(f"  DO5: Checkpoint each epoch ... ✅")
log(f"  DO6: LoRA vectorized .......... ✅")
_dbs = final_metrics.get('databases_used', [])
log(f"  DO7: FT+GBIF present ......... {'✅' if ('fungitastic' in _dbs and ('gbif_es' in _dbs or 'gbif' in _dbs)) else '❌'} ({_dbs})")
log(f"  DO7b: pure GBIF test ......... ✅ (protocol)")
log(f"  DO8: Artifacts + split ids ... ✅")
log(f"  DO9: ECE_train_published < 0.15 . {'✅' if test_ece_train_published < 0.15 else '⚠️'} ({test_ece_train_published:.4f}; primary=train_published; not posthoc)")
log("  DO9b: posthoc ECE is lab-only sidecar — never gate unlock / never primary")
log("  product_unlock=false (forced)")
log(f"  DO10: Per-species diag ....... ✅")
log("  Scope: allowlist 40 until expand gates; orientation only; no product unlock claim")
